# Section 1 — Environment Setup and Dependency Verification

This section installs the required dependencies for the Santali → English vocabulary-extension and LoRA experiments.

We pin `transformers==4.46.3` and `tokenizers==0.20.3` to maintain a consistent environment with the intended IndicTrans2 implementation.

After installation, the Kaggle session must be restarted before importing the libraries. We will then verify the installed versions and GPU availability.

**Important:** Do not load the model, tokenizer, or dataset in this section. Those tasks belong to later sections.

In [ ]:
# Section 1.2 — Install Pinned Dependencies

%pip install -q --no-cache-dir \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "IndicTransToolkit==1.1.1" \
    peft accelerate sacrebleu sentencepiece kagglehub


 **MANDATORY STEP: Restart SessionClick Restart Session in Kaggle's top menu bar. This flushes Python's in-memory module cache while keeping the newly installed files on disk.**


In [1]:
# Section 1.3 — Environment and GPU Verification

import sys
import importlib.metadata as metadata
import torch

print("=== Package Version Verification ===")

packages = [
    "transformers",
    "tokenizers",
    "IndicTransToolkit",
    "torch",
    "peft",
    "accelerate",
    "sacrebleu",
    "sentencepiece",
]

for package in packages:
    try:
        print(f"{package:22s}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:22s}: NOT INSTALLED")

print("\n=== Runtime Information ===")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory / (1024**3)
    )
    print(f"GPU memory: {gpu_memory:.2f} GiB")

assert metadata.version("transformers") == "4.46.3"
assert metadata.version("tokenizers") == "0.20.3"

print("\nEnvironment verification passed.")

=== Package Version Verification ===
transformers          : 4.46.3
tokenizers            : 0.20.3
IndicTransToolkit     : 1.1.1
torch                 : 2.11.0+cu128
peft                  : 0.20.0
accelerate            : 1.14.0
sacrebleu             : 2.6.0
sentencepiece         : 0.2.2

=== Runtime Information ===
Python version: 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB

Environment verification passed.


## Section 2 — Load the Shared Preprocessed Dataset

This experiment uses the published `common-tribal-preprocessing` Kaggle Dataset as the single source of truth for all data splits.

We will load the existing Santali training, validation, and internal test splits without performing additional cleaning or splitting.

- **Training split:** used for LoRA fine-tuning.
- **Validation split:** used for monitoring training, early stopping, and selecting the best checkpoint.
- **Internal test split:** reserved for final evaluation after model selection.
- **Competition test data:** kept separate and not used for training or model selection.

We will verify the loaded files, column names, row counts, missing values, and duplicate source-target pairs before proceeding.

In [2]:
# Section 2.2 — Locate the Published Dataset

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Top-level input folders:")
for path in sorted(INPUT_ROOT.iterdir()):
    print(" -", path.name)

# Search recursively because Kaggle may mount datasets under /kaggle/input/datasets/
matches = [
    path
    for path in INPUT_ROOT.rglob("santali_train.parquet")
    if path.parent.name == "splits"
]

assert len(matches) == 1, (
    f"Expected one santali_train.parquet file, found {len(matches)}. "
    "Check the attached dataset and input directory."
)

SPLIT_DIR = matches[0].parent
DATASET_DIR = SPLIT_DIR.parent

print("\nDataset directory:", DATASET_DIR)
print("Split directory:", SPLIT_DIR)

print("\nAvailable split files:")
for path in sorted(SPLIT_DIR.iterdir()):
    print(" -", path.name)

Top-level input folders:
 - datasets

Dataset directory: /kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing
Split directory: /kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing/splits

Available split files:
 - bhili_test.csv
 - bhili_test.parquet
 - bhili_train.csv
 - bhili_train.parquet
 - bhili_validation.csv
 - bhili_validation.parquet
 - gondi_test.csv
 - gondi_test.parquet
 - gondi_train.csv
 - gondi_train.parquet
 - gondi_validation.csv
 - gondi_validation.parquet
 - mundari_test.csv
 - mundari_test.parquet
 - mundari_train.csv
 - mundari_train.parquet
 - mundari_validation.csv
 - mundari_validation.parquet
 - santali_test.csv
 - santali_test.parquet
 - santali_train.csv
 - santali_train.parquet
 - santali_validation.csv
 - santali_validation.parquet


## Section 3 — Data Validation and Split Integrity

Validate the preprocessed training, validation, and internal test splits for Bhili, Gondi, Mundari, and Santali.

We will check expected columns, data types, missing or empty sentences, duplicate source-target pairs, and overlap between splits. The saved data will not be modified.

All checks must pass before tokenization analysis begins. The internal test splits remain reserved for final evaluation.

In [3]:
# Section 3.2 — Validate All Languages and Splits

import pandas as pd

language_columns = {
    "bhili": ("Hindi", "Bhili"),
    "gondi": ("Hindi", "Gondi"),
    "mundari": ("Hindi", "Mundari"),
    "santali": ("English", "Santali"),
}

split_names = ("train", "validation", "test")

# Reuse SPLIT_DIR established in Section 2
all_data = {
    language: {
        split: pd.read_parquet(
            SPLIT_DIR / f"{language}_{split}.parquet"
        )
        for split in split_names
    }
    for language in language_columns
}

validation_results = []

for language, splits in all_data.items():
    source_col, target_col = language_columns[language]

    # Validate each split
    for split_name, df in splits.items():
        checks = {
            "expected_columns": (
                source_col in df.columns
                and target_col in df.columns
            ),
            "missing_values": (
                not df[[source_col, target_col]].isna().any().any()
            ),
            "empty_sentences": (
                not df[source_col].astype(str).str.strip().eq("").any()
                and not df[target_col].astype(str).str.strip().eq("").any()
            ),
            "string_types": (
                df[source_col].map(lambda x: isinstance(x, str)).all()
                and df[target_col].map(lambda x: isinstance(x, str)).all()
            ),
            "duplicate_pairs": (
                not df.duplicated(
                    subset=[source_col, target_col]
                ).any()
            ),
        }

        validation_results.append({
            "language": language,
            "split": split_name,
            "rows": len(df),
            **checks,
        })

    # Check exact source-target pair overlap between splits
    pair_sets = {
        split: set(
            map(
                tuple,
                df[[source_col, target_col]].itertuples(
                    index=False, name=None
                )
            )
        )
        for split, df in splits.items()
    }

    overlap_train_val = len(
        pair_sets["train"] & pair_sets["validation"]
    )
    overlap_train_test = len(
        pair_sets["train"] & pair_sets["test"]
    )
    overlap_val_test = len(
        pair_sets["validation"] & pair_sets["test"]
    )

    print(f"\n{language.upper()} — Exact pair overlap")
    print("Train ↔ Validation:", overlap_train_val)
    print("Train ↔ Test:", overlap_train_test)
    print("Validation ↔ Test:", overlap_val_test)

results_df = pd.DataFrame(validation_results)

print("\n=== Split Validation Results ===")
display(results_df)

check_columns = [
    "expected_columns",
    "missing_values",
    "empty_sentences",
    "string_types",
    "duplicate_pairs",
]

all_checks_passed = results_df[check_columns].all().all()

print("\nAll individual split checks passed:", all_checks_passed)

assert all_checks_passed, (
    "Validation failed. Inspect the results before proceeding."
)

print("Basic data validation passed.")


BHILI — Exact pair overlap
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

GONDI — Exact pair overlap
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

MUNDARI — Exact pair overlap
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

SANTALI — Exact pair overlap
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

=== Split Validation Results ===


,language,split,rows,expected_columns,missing_values,empty_sentences,string_types,duplicate_pairs
0,bhili,train,17821,True,True,True,True,True
1,bhili,validation,990,True,True,True,True,True
2,bhili,test,991,True,True,True,True,True
3,gondi,train,18000,True,True,True,True,True
4,gondi,validation,1000,True,True,True,True,True
5,gondi,test,1000,True,True,True,True,True
6,mundari,train,18000,True,True,True,True,True
7,mundari,validation,1000,True,True,True,True,True
8,mundari,test,1000,True,True,True,True,True
9,santali,train,17883,True,True,True,True,True



All individual split checks passed: True
Basic data validation passed.


## 4. Model Selection and Tokenization Research

### 4.1 Research Configuration

This section investigates whether vocabulary extension improves neural machine translation for low-resource Indian tribal languages. The experiments will compare a standard LoRA fine-tuning baseline against a vocabulary-extended LoRA model while keeping the data splits and other experimental settings consistent.

The target language directions are:

- Hindi → Bhili
- Hindi → Gondi
- Hindi → Mundari
- English → Santali

Two model-size categories will be investigated:

1. A larger approximately 1B-parameter model fine-tuned using LoRA.
2. A smaller distilled model to assess the trade-off between computational cost and translation quality.

Before training, we will verify language support, source and target tokenizers, vocabulary dimensions, and GPU memory requirements. Tokenization fertility will be measured across all four target languages, and the most suitable pilot setup will be selected before implementing vocabulary extension.

The main experimental comparison will use the same training and evaluation data, random seed, LoRA configuration, and training budget wherever technically possible. Validation performance will guide checkpoint selection, while the test set will be reserved for final evaluation.

In [4]:
# Section 4.1 — Environment and existing research objects

import torch
import transformers
import tokenizers
import pandas as pd

print("Python/PyTorch environment")
print("---------------------------")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GiB"
    )

print("\nExisting research objects")
print("-------------------------")

for name in ["all_data", "language_columns", "SPLIT_DIR",
             "tokenizer", "processor", "MODEL_NAME"]:
    print(f"{name}: {'available' if name in globals() else 'not defined'}")

if "all_data" in globals():
    print("\nDataset split sizes:")
    for language, splits in all_data.items():
        print(
            f"{language}:",
            {split: len(df) for split, df in splits.items()}
        )

if "MODEL_NAME" in globals():
    print("\nPreviously loaded model checkpoint:", MODEL_NAME)

Python/PyTorch environment
---------------------------
PyTorch: 2.11.0+cu128
Transformers: 4.46.3
Tokenizers: 0.20.3
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB

Existing research objects
-------------------------
all_data: available
language_columns: available
SPLIT_DIR: available
tokenizer: not defined
processor: not defined
MODEL_NAME: not defined

Dataset split sizes:
bhili: {'train': 17821, 'validation': 990, 'test': 991}
gondi: {'train': 18000, 'validation': 1000, 'test': 1000}
mundari: {'train': 18000, 'validation': 1000, 'test': 1000}
santali: {'train': 17883, 'validation': 993, 'test': 994}


### 4.2 Dataset Structure and Translation Directions

The experiments use four language pairs: Hindi–Bhili, Hindi–Gondi, Hindi–Mundari, and English–Santali. The source and target columns must be verified before selecting a model because the translation direction determines which tokenizer and language configuration are appropriate.

We will also verify that all three data splits contain the expected columns and that the existing dataset objects are consistent. This step reuses the previously loaded data rather than reading the files again.

In [5]:
# Section 4.2 — Verify dataset columns and translation directions

expected_splits = {"train", "validation", "test"}

print("Dataset direction and schema verification")
print("=" * 55)

for language, (source_col, target_col) in language_columns.items():
    splits = all_data[language]

    assert set(splits.keys()) == expected_splits, (
        f"{language}: unexpected split names {list(splits.keys())}"
    )

    print(f"\nLanguage: {language.upper()}")
    print(f"Translation direction: {source_col} -> {target_col}")

    for split_name in ("train", "validation", "test"):
        df = splits[split_name]

        missing_cols = {
            source_col, target_col
        } - set(df.columns)

        assert not missing_cols, (
            f"{language}/{split_name}: missing columns {missing_cols}"
        )

        assert df[[source_col, target_col]].notna().all().all(), (
            f"{language}/{split_name}: missing source or target text"
        )

        print(
            f"  {split_name:10s}: {len(df):5d} rows | "
            f"source={source_col}, target={target_col}"
        )

    # Display one example to verify the actual direction.
    example = splits["train"].iloc[0]
    print("\n  Example source:", str(example[source_col])[:200])
    print("  Example target:", str(example[target_col])[:200])

print("\nAll dataset direction and schema checks passed.")

Dataset direction and schema verification

Language: BHILI
Translation direction: Hindi -> Bhili
  train     : 17821 rows | source=Hindi, target=Bhili
  validation:   990 rows | source=Hindi, target=Bhili
  test      :   991 rows | source=Hindi, target=Bhili

  Example source: युवा उद्यमी, जो किसी स्टार्टअप की योजना बना रहे हों, वे समग्र स्वास्थ्य सेवाओं में कई अवसर प्राप्त कर सकते हैं।
  Example target: खुद नो नीजी काम काज करवा वाळा जोवान मनखा ,ने कईनु काम सलाडवा नी ओजना बणाविरा से।, तिहुंनें हगळी स्वास्थ्य से।वायु मां घणा मौका मळी सके से। ।

Language: GONDI
Translation direction: Hindi -> Gondi
  train     : 18000 rows | source=Hindi, target=Gondi
  validation:  1000 rows | source=Hindi, target=Gondi
  test      :  1000 rows | source=Hindi, target=Gondi

  Example source: le; fcrrs le; ugha yxrk] ekbZ ds cPps cM+s gksus yxs
  Example target: समम बीते मासोर हन्दयामानांग पीलांग बेर्रससोर हन्दुंग

Language: MUNDARI
Translation direction: Hindi -> Mundari
  train     : 18000 rows | sourc

### 4.3 Candidate Model Checkpoints

We will investigate both the larger IndicTrans2 checkpoint family and the smaller distilled variants.

The purpose is to identify checkpoints that are appropriate for the required translation directions and feasible on a Tesla T4 GPU. LoRA reduces the number of trainable parameters, but it does not remove the memory cost of the frozen base model.

Candidate families include:

- Indic-to-English 1B: the checkpoint currently loaded in the notebook, retained for comparison but not assumed to be suitable.
- English-to-Indic 1B: a candidate for English-to-Santali, subject to language support.
- Indic-to-Indic 1B: a candidate for Hindi-to-Bhili, Gondi, and Mundari, subject to language support.
- Distilled English-to-Indic and Indic-to-Indic checkpoints: smaller alternatives for feasibility and performance comparisons.

Before selecting a checkpoint, we will inspect its configuration, vocabulary dimensions, architecture, and declared language metadata. A checkpoint's general Indic-language capability does not automatically imply support for every tribal language or script in this dataset.

In [9]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face authentication restored.")

Hugging Face authentication restored.


In [10]:
# Section 4.3 — Inspect candidate checkpoint configurations

from transformers import AutoConfig

CANDIDATE_MODELS = {
    "Indic -> English 1B": "ai4bharat/indictrans2-indic-en-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic distilled": "ai4bharat/indictrans2-en-indic-dist-200M",
    "Indic -> Indic distilled": "ai4bharat/indictrans2-indic-indic-dist-320M",
}

config_results = {}

for label, model_id in CANDIDATE_MODELS.items():
    print(f"\n{'=' * 70}\n{label}\n{model_id}")

    try:
        config = AutoConfig.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
        )

        config_results[label] = config

        print("Model type:", getattr(config, "model_type", None))
        print("Architecture:", getattr(config, "architectures", None))
        print("Encoder-decoder:", getattr(config, "is_encoder_decoder", None))
        print("Source vocab size:", getattr(config, "vocab_size", None))
        print("Target vocab size:", getattr(config, "tgt_vocab_size", None))
        print("Hidden size:", getattr(config, "d_model", None))
        print("Encoder layers:", getattr(config, "encoder_layers", None))
        print("Decoder layers:", getattr(config, "decoder_layers", None))

        # Print likely language metadata fields if present.
        for key in ("src_lang", "tgt_lang", "languages", "lang_code_to_id"):
            value = getattr(config, key, None)
            if value is not None:
                if isinstance(value, dict) and len(value) > 20:
                    value = f"<mapping with {len(value)} entries>"
                print(f"{key}:", value)

    except Exception as e:
        print("CONFIG INSPECTION FAILED:", type(e).__name__, str(e)[:400])

print("\nConfiguration inspection complete.")


Indic -> English 1B
ai4bharat/indictrans2-indic-en-1B


config.json: 0.00B [00:00, ?B/s]

configuration_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Model type: IndicTrans
Architecture: ['IndicTransForConditionalGeneration']
Encoder-decoder: True
Source vocab size: 32296
Target vocab size: None
Hidden size: None
Encoder layers: 18
Decoder layers: 18

English -> Indic 1B
ai4bharat/indictrans2-en-indic-1B


...

Model type: IndicTrans
Architecture: ['IndicTransForConditionalGeneration']
Encoder-decoder: True
Source vocab size: 122672
Target vocab size: None
Hidden size: None
Encoder layers: 18
Decoder layers: 18

Indic -> Indic distilled
ai4bharat/indictrans2-indic-indic-dist-320M


config.json: 0.00B [00:00, ?B/s]

configuration_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-dist-320M:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Model type: IndicTrans
Architecture: ['IndicTransForConditionalGeneration']
Encoder-decoder: True
Source vocab size: 122672
Target vocab size: None
Hidden size: None
Encoder layers: 18
Decoder layers: 18

Configuration inspection complete.


#### 4.3.2 Compare Model Architecture and Vocabulary Configuration

The candidate checkpoints are now accessible. We will compare their configuration fields, including model dimensions, encoder and decoder settings, vocabulary sizes, and parameter-count metadata where available.

This comparison will help us identify the differences between the larger and distilled models. We will also check whether each checkpoint uses separate source and target vocabularies, since that distinction is critical for vocabulary extension.

Configuration metadata is preliminary evidence only. The final decision will also depend on tokenizer behavior, language support, memory requirements, and the ability to fine-tune the model reliably.

In [11]:
# Section 4.3.2 — Compare available configuration fields

from transformers import AutoConfig

fields_to_compare = [
    "d_model",
    "hidden_size",
    "encoder_embed_dim",
    "decoder_embed_dim",
    "encoder_ffn_dim",
    "decoder_ffn_dim",
    "encoder_layers",
    "decoder_layers",
    "encoder_attention_heads",
    "decoder_attention_heads",
    "vocab_size",
    "tgt_vocab_size",
    "max_position_embeddings",
    "tie_word_embeddings",
]

comparison_rows = []

for label, model_id in CANDIDATE_MODELS.items():
    config = config_results.get(label)

    if config is None:
        try:
            config = AutoConfig.from_pretrained(
                model_id,
                trust_remote_code=True,
                token=hf_token,
            )
            config_results[label] = config
        except Exception as e:
            print(f"Could not inspect {label}: {type(e).__name__}: {e}")
            continue

    row = {
        "candidate": label,
        "model_id": model_id,
    }

    for field in fields_to_compare:
        value = getattr(config, field, None)
        if isinstance(value, (list, dict)):
            value = f"<{type(value).__name__}: {len(value)} items>"
        row[field] = value

    comparison_rows.append(row)

config_comparison = pd.DataFrame(comparison_rows).set_index("candidate")
display(config_comparison.T)

candidate,Indic -> English 1B,English -> Indic 1B,Indic -> Indic 1B,English -> Indic distilled,Indic -> Indic distilled
model_id,ai4bharat/indictrans2-indic-en-1B,ai4bharat/indictrans2-en-indic-1B,ai4bharat/indictrans2-indic-indic-1B,ai4bharat/indictrans2-en-indic-dist-200M,ai4bharat/indictrans2-indic-indic-dist-320M
d_model,None,None,None,None,None
hidden_size,None,None,None,None,None
encoder_embed_dim,1024,1024,1024,512,512
decoder_embed_dim,1024,1024,1024,512,512
encoder_ffn_dim,8192,8192,8192,2048,2048
decoder_ffn_dim,8192,8192,8192,2048,2048
encoder_layers,18,18,18,18,18
decoder_layers,18,18,18,18,18
encoder_attention_heads,16,16,16,8,8


### 4.4 Tokenizer and Language Configuration

Before measuring tokenization fertility, we must identify the tokenizer associated with each candidate checkpoint.

IndicTrans2 uses language-specific source and target processing. Therefore, we must not assume that a single SentencePiece vocabulary or a generic tokenizer interface fully describes both sides of the translation model.

For each accessible checkpoint, we will inspect:

- Tokenizer class and vocabulary size.
- Source and target SentencePiece vocabulary sizes, where available.
- Special-token IDs.
- Language-code handling.
- Whether the tokenizer can be initialized without loading the model weights.

These checks will help establish which tokenizer is appropriate for Hindi-to-tribal and English-to-Santali translation, and what changes vocabulary extension would require.

In [12]:
# Section 4.4 — Inspect candidate tokenizers

from transformers import AutoTokenizer

tokenizer_results = {}

for label, model_id in CANDIDATE_MODELS.items():
    print(f"\n{'=' * 70}")
    print(label)
    print(model_id)

    try:
        tok = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
        )
        tokenizer_results[label] = tok

        print("Tokenizer class:", type(tok).__name__)
        print("len(tokenizer):", len(tok))
        print("Tokenizer vocab_size:", getattr(tok, "vocab_size", None))
        print("Special tokens:", tok.special_tokens_map)

        for attr in ("src_spm", "tgt_spm"):
            spm = getattr(tok, attr, None)
            if spm is not None:
                try:
                    print(f"{attr} piece count:", spm.GetPieceSize())
                except Exception:
                    print(f"{attr}: available, piece count unavailable")
            else:
                print(f"{attr}: not exposed")

        print("Special token IDs:")
        for token_name in (
            "bos_token_id",
            "eos_token_id",
            "pad_token_id",
            "unk_token_id",
        ):
            print(f"  {token_name}:", getattr(tok, token_name, None))

        # Inspect language-related tokenizer attributes without assuming
        # a particular custom-tokenizer implementation.
        for attr in ("lang_code_to_id", "supported_languages", "src_lang", "tgt_lang"):
            value = getattr(tok, attr, None)
            if value is not None:
                if isinstance(value, dict):
                    print(f"{attr}: {len(value)} entries")
                    print("  Sample:", list(value.items())[:8])
                else:
                    print(f"{attr}:", value)

    except Exception as e:
        print("TOKENIZER INSPECTION FAILED:", type(e).__name__, str(e)[:350])


Indic -> English 1B
ai4bharat/indictrans2-indic-en-1B


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Tokenizer class: IndicTransTokenizer
len(tokenizer): 122706
Tokenizer vocab_size: 122706
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
src_spm piece count: 128000
tgt_spm piece count: 128000
Special token IDs:
  bos_token_id: 0
  eos_token_id: 2
  pad_token_id: 1
  unk_token_id: 3


### 4.5 Tokenization Fertility Across Tribal Languages

Tokenization fertility measures how many subword tokens a tokenizer produces for a given amount of text. High fertility can indicate that words are fragmented into many small pieces, potentially making training less efficient.

We will evaluate the source and target SentencePiece models separately on the tribal-language text for Bhili, Gondi, Mundari, and Santali. This is a diagnostic comparison: the source tokenizer applied to a target-language sentence is not necessarily the tokenizer the translation model would use during normal inference.

For each language and tokenizer, we will calculate:

- Average tokens per sentence.
- Average tokens per whitespace-separated word.
- Word split rate: the percentage of whitespace-separated words represented by more than one SentencePiece token.
- Unknown-token rate, where the tokenizer exposes an unknown-token ID.

A fixed random seed and sample size will be used to make the comparisons reproducible. These measurements will help identify vocabulary gaps, but they do not independently establish translation quality.

In [13]:
# Section 4.5 — Compare source and target SentencePiece fertility

import numpy as np
import pandas as pd

SAMPLE_SIZE = 2000
RANDOM_SEED = 42

fertility_rows = []

for language, (_, target_col) in language_columns.items():
    df = all_data[language]["train"]
    texts = (
        df[target_col]
        .dropna()
        .astype(str)
        .sample(
            n=min(SAMPLE_SIZE, len(df)),
            random_state=RANDOM_SEED,
        )
        .tolist()
    )

    # Both models expose these attributes through IndicTransTokenizer.
    # This compares the SP models diagnostically on the same text.
    for model_label in [
        "Indic -> English 1B",
        "English -> Indic 1B",
        "Indic -> Indic 1B",
        "English -> Indic distilled",
        "Indic -> Indic distilled",
    ]:
        tok = tokenizer_results[model_label]

        for spm_name in ["src_spm", "tgt_spm"]:
            spm = getattr(tok, spm_name, None)
            if spm is None:
                continue

            sentence_token_counts = []
            total_words = 0
            total_word_tokens = 0
            split_words = 0
            unknown_pieces = 0
            total_pieces = 0

            for text in texts:
                pieces = spm.EncodeAsPieces(text)
                sentence_token_counts.append(len(pieces))
                total_pieces += len(pieces)

                words = text.split()
                total_words += len(words)

                for word in words:
                    word_pieces = spm.EncodeAsPieces(word)
                    total_word_tokens += len(word_pieces)

                    if len(word_pieces) > 1:
                        split_words += 1

                    unknown_pieces += sum(
                        piece == "<unk>" for piece in word_pieces
                    )

            fertility_rows.append({
                "language": language,
                "text_column": target_col,
                "model": model_label,
                "spm": spm_name,
                "sentences": len(texts),
                "avg_tokens_per_sentence": round(
                    np.mean(sentence_token_counts), 3
                ),
                "tokens_per_word": round(
                    total_word_tokens / max(total_words, 1), 3
                ),
                "word_split_rate_pct": round(
                    100 * split_words / max(total_words, 1), 3
                ),
                "unknown_piece_rate_pct": round(
                    100 * unknown_pieces / max(total_pieces, 1), 4
                ),
            })

fertility_df = pd.DataFrame(fertility_rows)

display(
    fertility_df.sort_values(
        ["language", "model", "spm"]
    ).reset_index(drop=True)
)

,language,text_column,model,spm,sentences,avg_tokens_per_sentence,tokens_per_word,word_split_rate_pct,unknown_piece_rate_pct
0,bhili,Bhili,English -> Indic 1B,src_spm,2000,107.440,4.982,98.349,0.0
1,bhili,Bhili,English -> Indic 1B,tgt_spm,2000,31.243,1.449,32.004,0.0
2,bhili,Bhili,English -> Indic distilled,src_spm,2000,107.440,4.982,98.349,0.0
3,bhili,Bhili,English -> Indic distilled,tgt_spm,2000,31.243,1.449,32.004,0.0
4,bhili,Bhili,Indic -> English 1B,src_spm,2000,31.243,1.449,32.004,0.0
5,bhili,Bhili,Indic -> English 1B,tgt_spm,2000,107.440,4.982,98.349,0.0
6,bhili,Bhili,Indic -> Indic 1B,src_spm,2000,31.243,1.449,32.004,0.0
7,bhili,Bhili,Indic -> Indic 1B,tgt_spm,2000,31.243,1.449,32.004,0.0
8,bhili,Bhili,Indic -> Indic distilled,src_spm,2000,31.243,1.449,32.004,0.0
9,bhili,Bhili,Indic -> Indic distilled,tgt_spm,2000,31.243,1.449,32.004,0.0


### 4.5.1 Identifying Distinct SentencePiece Vocabularies

The previous fertility table contains repeated measurements across model
checkpoints. However, tokenizer vocabularies cannot be considered identical
merely because they have the same number of pieces.

We will compute a SHA-256 fingerprint from the ordered SentencePiece
vocabulary of each source and target tokenizer. Matching fingerprints
indicate identical ordered piece lists.

This allows us to identify genuinely distinct vocabularies before comparing
tokenization fertility across the four target languages.

In [14]:
# Section 4.5.1 — Fingerprint actual SentencePiece vocabularies

import hashlib
import pandas as pd

spm_fingerprints = {}

for model_name, tok in tokenizer_results.items():
    for spm_name in ("src_spm", "tgt_spm"):
        spm = getattr(tok, spm_name)
        piece_count = spm.GetPieceSize()

        hasher = hashlib.sha256()

        for piece_id in range(piece_count):
            piece = spm.IdToPiece(piece_id)
            hasher.update(piece.encode("utf-8"))
            hasher.update(b"\n")

        fingerprint = hasher.hexdigest()

        spm_fingerprints[(model_name, spm_name)] = {
            "model": model_name,
            "spm": spm_name,
            "piece_count": piece_count,
            "fingerprint": fingerprint,
        }

fingerprint_df = pd.DataFrame(spm_fingerprints.values())

display(
    fingerprint_df[
        ["model", "spm", "piece_count", "fingerprint"]
    ].sort_values(["piece_count", "model", "spm"]).reset_index(drop=True)
)

print(
    "\nNumber of distinct SentencePiece vocabularies:",
    fingerprint_df["fingerprint"].nunique()
)

print("\nVocabulary groups:")
display(
    fingerprint_df.groupby(
        ["piece_count", "fingerprint"], as_index=False
    ).agg(
        models=("model", lambda x: ", ".join(x)),
        tokenizer_sides=("spm", lambda x: ", ".join(x)),
        occurrences=("model", "size"),
    ).sort_values(["piece_count", "occurrences"], ascending=[True, False])
)

,model,spm,piece_count,fingerprint
0,English -> Indic 1B,src_spm,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...
1,English -> Indic distilled,src_spm,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...
2,Indic -> English 1B,tgt_spm,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...
3,English -> Indic 1B,tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
4,English -> Indic distilled,tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
5,Indic -> English 1B,src_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
6,Indic -> Indic 1B,src_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
7,Indic -> Indic 1B,tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
8,Indic -> Indic distilled,src_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
9,Indic -> Indic distilled,tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...



Number of distinct SentencePiece vocabularies: 2

Vocabulary groups:


,piece_count,fingerprint,models,tokenizer_sides,occurrences
0,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...,"Indic -> English 1B, English -> Indic 1B, Engl...","tgt_spm, src_spm, src_spm",3
1,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...,"Indic -> English 1B, English -> Indic 1B, Indi...","src_spm, tgt_spm, src_spm, tgt_spm, tgt_spm, s...",7


### 4.5.2 Comparing Distinct SentencePiece Vocabularies

The fingerprint analysis identified two distinct SentencePiece vocabularies:
one containing 32,000 pieces and another containing 128,000 pieces.

We will now compare their tokenization behavior on the target-language
sentences. This comparison is diagnostic: lower fertility may indicate
more efficient tokenization, but it does not by itself establish better
translation quality or model compatibility.

In [15]:
# Section 4.5.2 — Compare the two distinct vocabularies

import pandas as pd
import numpy as np

# Select one representative tokenizer for each actual fingerprint.
unique_fingerprints = fingerprint_df.drop_duplicates(
    subset=["fingerprint"]
)

spm_32k = None
spm_128k = None

for _, row in unique_fingerprints.iterrows():
    tok = tokenizer_results[row["model"]]
    spm = getattr(tok, row["spm"])
    piece_count = spm.GetPieceSize()

    if piece_count == 32000:
        spm_32k = spm
    elif piece_count == 128000:
        spm_128k = spm

assert spm_32k is not None, "Could not locate the 32k vocabulary."
assert spm_128k is not None, "Could not locate the 128k vocabulary."

rows = []

for language, (_, target_col) in language_columns.items():
    texts = (
        all_data[language]["train"][target_col]
        .dropna()
        .astype(str)
        .head(2000)
        .tolist()
    )

    for vocab_size, spm in [
        (32000, spm_32k),
        (128000, spm_128k),
    ]:
        total_tokens = 0
        total_words = 0
        split_words = 0

        for text in texts:
            total_tokens += len(spm.EncodeAsPieces(text))

            for word in text.split():
                total_words += 1
                if len(spm.EncodeAsPieces(word)) > 1:
                    split_words += 1

        rows.append({
            "language": language,
            "vocabulary_size": vocab_size,
            "sentences": len(texts),
            "avg_tokens_per_sentence": round(
                total_tokens / max(len(texts), 1), 3
            ),
            "tokens_per_word": round(
                total_tokens / max(total_words, 1), 3
            ),
            "word_split_rate_pct": round(
                100 * split_words / max(total_words, 1), 3
            ),
        })

comparison_df = pd.DataFrame(rows)

display(
    comparison_df.sort_values(
        ["language", "vocabulary_size"]
    ).reset_index(drop=True)
)

,language,vocabulary_size,sentences,avg_tokens_per_sentence,tokens_per_word,word_split_rate_pct
0,bhili,32000,2000,105.506,4.957,98.264
1,bhili,128000,2000,30.831,1.448,32.350
2,gondi,32000,2000,74.555,5.532,99.547
3,gondi,128000,2000,23.679,1.757,48.418
4,mundari,32000,2000,78.115,5.530,97.795
5,mundari,128000,2000,30.438,2.155,62.959
6,santali,32000,2000,39.544,2.029,99.574
7,santali,128000,2000,28.102,1.442,27.239


In [21]:
# Section 4.6 — Check existing tokenizer objects and language-code support

print("Candidate tokenizers available:", len(tokenizer_results))
print("Candidate checkpoints:", list(tokenizer_results.keys()))

print("\nLanguage-related tokenizer attributes:")
for label, tok in tokenizer_results.items():
    print(f"\n{label}")

    for attr in dir(tok):
        if any(term in attr.lower() for term in ("lang_code", "supported_languages")):
            try:
                value = getattr(tok, attr)
                if not callable(value):
                    print(f"  {attr}: {str(value)[:300]}")
            except Exception:
                pass

print("\nProcessor class:", type(processor).__name__)

Candidate tokenizers available: 5
Candidate checkpoints: ['Indic -> English 1B', 'English -> Indic 1B', 'Indic -> Indic 1B', 'English -> Indic distilled', 'Indic -> Indic distilled']

Language-related tokenizer attributes:

Indic -> English 1B

English -> Indic 1B

Indic -> Indic 1B

English -> Indic distilled

Indic -> Indic distilled

Processor class: IndicProcessor


In [23]:
# Section 4.6 — Inspect the compiled IndicProcessor API

import inspect

print("Processor methods and signatures:\n")

for name in dir(processor):
    if name.startswith("_"):
        continue

    try:
        obj = getattr(processor, name)
        if callable(obj):
            try:
                signature = inspect.signature(obj)
            except (TypeError, ValueError):
                signature = "(signature unavailable)"

            print(f"{name}{signature}")
    except Exception as e:
        print(f"{name}: inspection failed ({type(e).__name__})")

Processor methods and signatures:

postprocess_batch(sents, lang='hin_Deva', visualize=False, num_return_sequences=1)
preprocess_batch(batch, src_lang, tgt_lang=None, is_target=False, visualize=False)


In [24]:
# Section 4.6 — Search installed toolkit for language-code definitions

from pathlib import Path
import IndicTransToolkit

package_dir = Path(IndicTransToolkit.__file__).parent

print("Toolkit directory:", package_dir)

keywords = ("bhili", "bhili", "gondi", "mundari", "santali",
            "hin_deva", "eng_latn", "language_codes")

matches = []

for path in package_dir.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".py", ".json", ".txt", ".yaml", ".yml"}:
        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
            found = [word for word in keywords if word in content.lower()]
            if found:
                matches.append((str(path.relative_to(package_dir)), found))
        except OSError:
            pass

if matches:
    for filename, found in matches:
        print(f"{filename}: {', '.join(sorted(set(found)))}")
else:
    print("No matching text files found in the installed toolkit.")

Toolkit directory: /usr/local/lib/python3.13/dist-packages/IndicTransToolkit
evaluator.py: eng_latn, hin_deva


In [25]:
# Section 4.6 — Inspect checkpoint language metadata

for name, config in config_results.items():
    print(f"\n{'=' * 60}")
    print(name)

    if isinstance(config, dict):
        for key in ("model_name", "name_or_path", "architectures",
                    "model_type", "language_codes", "langs",
                    "lang_code_to_id", "id_to_lang_code"):
            if key in config:
                print(f"{key}: {config[key]}")
    else:
        print("Configuration object:", type(config).__name__)
        for key in ("name_or_path", "architectures", "model_type",
                    "language_codes", "langs",
                    "lang_code_to_id", "id_to_lang_code"):
            value = getattr(config, key, None)
            if value is not None:
                print(f"{key}: {value}")


Indic -> English 1B
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-indic-en-1B
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

English -> Indic 1B
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-en-indic-1B
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

Indic -> Indic 1B
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-indic-indic-1B
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

English -> Indic distilled
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-en-indic-dist-200M
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

Indic -> Indic distilled
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-indic-indic-dist-320M
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans


In [26]:
# Section 4.6 — Inspect tokenizer API for language handling

for name, tok in tokenizer_results.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("Class:", type(tok).__name__)

    for attr in dir(tok):
        if any(term in attr.lower() for term in (
            "lang", "language", "src_spm", "tgt_spm",
            "preprocess", "special"
        )):
            try:
                value = getattr(tok, attr)
                if not callable(value):
                    print(f"{attr}: {str(value)[:200]}")
            except Exception:
                pass


Indic -> English 1B
Class: IndicTransTokenizer
SPECIAL_TOKENS_ATTRIBUTES: ['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']
_additional_special_tokens: []
additional_special_tokens: []
additional_special_tokens_ids: []
all_special_ids: [0, 2, 3, 1]
all_special_tokens: ['<s>', '</s>', '<unk>', '<pad>']
all_special_tokens_extended: [AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), Ad
special_tokens_map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
special_tokens_map_extended: {'bos_token': AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalize
split_special_tokens: False
src_spm: <sentencepiece.Senten

In [27]:
# Section 4.6 — Prepare one example per translation task
# This cell only inspects the existing data; it does not train or load models.

for language, (src_col, tgt_col) in language_columns.items():
    example = all_data[language]["validation"].iloc[0]

    print(f"\n{'=' * 55}")
    print(f"Task: {src_col} -> {tgt_col}")
    print("Language key:", language)
    print("Source column:", src_col)
    print("Target column:", tgt_col)
    print("Source example:", str(example[src_col])[:250])
    print("Target example:", str(example[tgt_col])[:250])


Task: Hindi -> Bhili
Language key: bhili
Source column: Hindi
Target column: Bhili
Source example: अंग्रेजी राज्य की आधिकारिक भाषा है।
Target example: अंग्रेजी राज्य नी अधिकारिक भाषा से ।

Task: Hindi -> Gondi
Language key: gondi
Source column: Hindi
Target column: Gondi
Source example: सरकारी संस्थानों में आरटीआई को लेकर प्रशिक्षण कार्यक्रम आयोजित किए जाते हैं
Target example: सरकारी डेटा ने आर टी आई तुन ओस प्रशिक्षण कार्यक्रम शुरू कियाना आयता

Task: Hindi -> Mundari
Language key: mundari
Source column: Hindi
Target column: Mundari
Source example: और इसलिए सामान्‍य नागरिक - शहर हो या गांव - उसकी मेहनत का कोई मुकाबला नहीं है।
Target example: ओड़ोः एनामेनते सामान्य होड़ोको शहर तयु च हातु - इनिआ कामि रेओ जेतन मुकाबला बनोः ।

Task: English -> Santali
Language key: santali
Source column: English
Target column: Santali
Source example: According to Minhaj, Iltutmish re-asserted Delhi's control over Badaun, Awadh, Banaras, and Siwalik in a series of campaigns.
Target example: ᱢᱤᱱᱦᱟᱡ ᱟᱜ ᱠᱟᱛᱷᱟᱛ

# Section 4.6 results and analysis — Language and Model Compatibility Analysis

## 1. Research Objective

Our research investigates whether vocabulary extension and informed embedding initialization can improve neural machine translation for four low-resource Indian tribal languages: Bhili, Gondi, Mundari, and Santali.

We will compare two approaches:

1. **Baseline:** Fine-tune a pretrained IndicTrans2 model using LoRA while retaining its original vocabulary.
2. **Proposed method:** Fine-tune the same model using LoRA after extending its vocabulary and applying a selected embedding-initialization strategy.

For a fair comparison, both approaches must use the same checkpoint, training data, data splits, evaluation procedure, and comparable training settings.

## 2. Translation Tasks and Dataset Examples

The prepared dataset contains four translation tasks:

| Language | Translation direction | Source language | Target language |
|---|---|---|---|
| Bhili | Hindi → Bhili | Hindi | Bhili |
| Gondi | Hindi → Gondi | Hindi | Gondi |
| Mundari | Hindi → Mundari | Hindi | Mundari |
| Santali | English → Santali | English | Santali |

The validation examples inspected in the notebook confirm that the expected source and target columns are available.

The Santali target example uses the **Ol Chiki script**. The Bhili, Gondi, and Mundari examples use their respective dataset representations, which include Devanagari-script text.

These examples confirm the dataset structure, but do not establish translation quality or pretrained-model compatibility.

## 3. Findings from the Official Language Information

The IndicTrans2 language information examined during this investigation lists Santali with the language code `sat_Olck`, corresponding to the Ol Chiki script.

Bhili, Gondi, and Mundari were not found in the supported-language lists examined for the candidate IndicTrans2 checkpoints.

This leads to the following distinction:

- **Santali:** A listed language, with a documented script-specific language code. We still need to verify the complete inference pipeline for the selected checkpoint.
- **Bhili, Gondi, and Mundari:** Languages for which we must investigate adaptation beyond the officially listed language coverage. We must not assume that inventing or assigning language codes will make these languages supported.

The distinction is important because tokenizer compatibility, language-code acceptance, and actual translation capability are separate properties.

## 4. Candidate Checkpoints

Five IndicTrans2 checkpoints were inspected.

| Checkpoint | Approximate model size | Intended translation direction |
|---|---:|---|
| `ai4bharat/indictrans2-indic-en-1B` | 1B | Indic → English |
| `ai4bharat/indictrans2-en-indic-1B` | 1B | English → Indic |
| `ai4bharat/indictrans2-indic-indic-1B` | 1B | Indic → Indic |
| `ai4bharat/indictrans2-en-indic-dist-200M` | 200M | English → Indic |
| `ai4bharat/indictrans2-indic-indic-dist-320M` | 320M | Indic → Indic |

All five inspected configurations use the `IndicTransForConditionalGeneration` architecture and have an encoder-decoder structure.

The 1B checkpoints use 1024-dimensional encoder and decoder embeddings, 8192-dimensional feed-forward layers, 18 encoder layers, 18 decoder layers, and 16 attention heads.

The distilled checkpoints use 512-dimensional embeddings, 2048-dimensional feed-forward layers, 18 encoder layers, 18 decoder layers, and 8 attention heads.

These differences make model size a relevant experimental variable. A larger checkpoint may offer greater capacity, but its

In [28]:
# Section 4.6 — Smoke-test preprocessing for supported-language examples
# This checks text preparation only, not translation quality.

preprocessing_tests = [
    {
        "task": "English -> Santali",
        "src_lang": "eng_Latn",
        "tgt_lang": "sat_Olck",
        "source": all_data["santali"]["validation"].iloc[0]["English"],
        "target": all_data["santali"]["validation"].iloc[0]["Santali"],
    },
    {
        "task": "Hindi source preprocessing",
        "src_lang": "hin_Deva",
        "tgt_lang": None,
        "source": all_data["bhili"]["validation"].iloc[0]["Hindi"],
        "target": None,
    },
]

for test in preprocessing_tests:
    print(f"\n{'=' * 55}")
    print("Test:", test["task"])

    try:
        processed = processor.preprocess_batch(
            [test["source"]],
            src_lang=test["src_lang"],
            tgt_lang=test["tgt_lang"],
        )
        print("Preprocessing succeeded.")
        print("Result:", processed[0][:300])
    except Exception as exc:
        print("Preprocessing failed:", type(exc).__name__, str(exc)[:500])


Test: English -> Santali
Preprocessing succeeded.
Result: eng_Latn sat_Olck According to Minhaj , Iltutmish re-asserted Delhi 's control over Badaun , Awadh , Banaras , and Siwalik in a series of campaigns .

Test: Hindi source preprocessing
Preprocessing succeeded.
Result: hin_Deva None अंग्रेजी राज्य की आधिकारिक भाषा है ।


In [29]:
# Section 4.6 — Santali tokenizer round-trip check
# Reuses the tokenizer already loaded in tokenizer_results.

tok = tokenizer_results["English -> Indic 1B"]

row = all_data["santali"]["validation"].iloc[0]
source_text = row["English"]
target_text = row["Santali"]

# Encode the source using the English source SentencePiece model
source_pieces = tok.src_spm.encode(source_text, out_type=str)

# Encode the Santali target using the target SentencePiece model
target_pieces = tok.tgt_spm.encode(target_text, out_type=str)

# Decode each sequence back to text
source_roundtrip = tok.src_spm.decode(source_pieces)
target_roundtrip = tok.tgt_spm.decode(target_pieces)

print("Source SentencePiece pieces:", len(source_pieces))
print("Target SentencePiece pieces:", len(target_pieces))

print("\nOriginal Santali target:")
print(target_text)

print("\nDecoded Santali target:")
print(target_roundtrip)

print("\nSantali round-trip exact match:", target_text == target_roundtrip)
print("Source round-trip exact match:", source_text == source_roundtrip)

Source SentencePiece pieces: 40
Target SentencePiece pieces: 39

Original Santali target:
ᱢᱤᱱᱦᱟᱡ ᱟᱜ ᱠᱟᱛᱷᱟᱛᱮ, ᱤᱞᱛᱩᱫᱢᱤᱥ ᱟᱹᱰᱤᱫᱷᱟᱣ ᱥᱟᱸᱜᱷᱟᱨ ᱨᱮ ᱵᱟᱫᱟᱩᱱ, ᱟᱣᱭᱟᱫᱷ, ᱵᱮᱱᱟᱨᱚᱥ ᱟᱨ ᱥᱤᱵᱟᱞᱤᱠ ᱪᱮᱛᱟᱱ ᱫᱚᱲᱦᱟᱛᱮ ᱫᱤᱞᱞᱤ ᱜᱚᱵᱚᱞᱨᱮ ᱛᱷᱟᱯᱚᱱ ᱞᱮᱫᱟᱭ ᱾

Decoded Santali target:
ᱢᱤᱱᱦᱟᱡ ᱟᱜ ᱠᱟᱛᱷᱟᱛᱮ, ᱤᱞᱛᱩᱫᱢᱤᱥ ᱟᱹᱰᱤᱫᱷᱟᱣ ᱥᱟᱸᱜᱷᱟᱨ ᱨᱮ ᱵᱟᱫᱟᱩᱱ, ᱟᱣᱭᱟᱫᱷ, ᱵᱮᱱᱟᱨᱚᱥ ᱟᱨ ᱥᱤᱵᱟᱞᱤᱠ ᱪᱮᱛᱟᱱ ᱫᱚᱲᱦᱟᱛᱮ ᱫᱤᱞᱞᱤ ᱜᱚᱵᱚᱞᱨᱮ ᱛᱷᱟᱯᱚᱱ ᱞᱮᱫᱟᱭ ᱾

Santali round-trip exact match: True
Source round-trip exact match: True


In [30]:
# Section 4.6 — Round-trip and tokenization check for all target languages

tokenizer_for_task = {
    "bhili": "Indic -> Indic 1B",
    "gondi": "Indic -> Indic 1B",
    "mundari": "Indic -> Indic 1B",
    "santali": "English -> Indic 1B",
}

for language, tokenizer_name in tokenizer_for_task.items():
    tok = tokenizer_results[tokenizer_name]
    _, target_col = language_columns[language]
    df = all_data[language]["validation"].head(100)

    token_counts = []
    exact_matches = 0

    for text in df[target_col].astype(str):
        pieces = tok.tgt_spm.encode(text, out_type=str)
        decoded = tok.tgt_spm.decode(pieces)

        token_counts.append(len(pieces))
        exact_matches += (decoded == text)

    print(f"\n{'=' * 55}")
    print(f"Language: {language}")
    print(f"Tokenizer: {tokenizer_name}")
    print(f"Examples checked: {len(df)}")
    print(f"Mean target SentencePiece tokens: {sum(token_counts) / len(token_counts):.2f}")
    print(f"Exact round trips: {exact_matches}/{len(df)}")


Language: bhili
Tokenizer: Indic -> Indic 1B
Examples checked: 100
Mean target SentencePiece tokens: 34.08
Exact round trips: 84/100

Language: gondi
Tokenizer: Indic -> Indic 1B
Examples checked: 100
Mean target SentencePiece tokens: 23.97
Exact round trips: 80/100

Language: mundari
Tokenizer: Indic -> Indic 1B
Examples checked: 100
Mean target SentencePiece tokens: 28.12
Exact round trips: 53/100

Language: santali
Tokenizer: English -> Indic 1B
Examples checked: 100
Mean target SentencePiece tokens: 27.23
Exact round trips: 97/100


In [31]:
# Section 4.6 — Inspect tokenizer round-trip failures

tokenizer_for_task = {
    "bhili": "Indic -> Indic 1B",
    "gondi": "Indic -> Indic 1B",
    "mundari": "Indic -> Indic 1B",
    "santali": "English -> Indic 1B",
}

for language, tokenizer_name in tokenizer_for_task.items():
    tok = tokenizer_results[tokenizer_name]
    _, target_col = language_columns[language]
    df = all_data[language]["validation"].head(100)

    failures = []

    for text in df[target_col].astype(str):
        pieces = tok.tgt_spm.encode(text, out_type=str)
        decoded = tok.tgt_spm.decode(pieces)

        if decoded != text:
            failures.append({
                "original": text,
                "decoded": decoded,
                "original_repr": repr(text),
                "decoded_repr": repr(decoded),
                "original_length": len(text),
                "decoded_length": len(decoded),
                "pieces": pieces,
            })

    print(f"\n{'=' * 70}")
    print(f"Language: {language} | Failures: {len(failures)}/100")

    for i, item in enumerate(failures[:3], start=1):
        print(f"\nFailure {i}")
        print("Original:     ", item["original"][:250])
        print("Decoded:      ", item["decoded"][:250])
        print("Original repr:", item["original_repr"][:300])
        print("Decoded repr: ", item["decoded_repr"][:300])
        print(
            "Lengths:      ",
            item["original_length"],
            "->",
            item["decoded_length"],
        )
        print("Pieces:       ", item["pieces"][:40])


Language: bhili | Failures: 16/100

Failure 1
Original:      बुडमण बंगाल नी पोलिस ने एक अगाड़ी नो पोलिस अधिकारी ने केदु की भारत अने बांगलादेश नी सेड़ा पर बदा धाणो को ननजर राखवा केदु से अने बदाी सेडा न जिलो को कई भी संखा थाई तेर मेरे जाई नजर रखवा केदु है ।
Decoded:       बुडमण बंगाल नी पोलिस ने एक अगाड़ी नो पोलिस अधिकारी ने केदु की भारत अने बांगलादेश नी सेड़ा पर बदा धाणो को ननजर राखवा केदु से अने बदाी सेडा न जिलो को कई भी संखा थाई तेर मेरे जाई नजर रखवा केदु है ।
Original repr: 'बुडमण बंगाल नी पोलिस ने एक अगाड़ी नो पोलिस अधिकारी ने केदु की भारत अने बांगलादेश नी सेड़ा पर बदा धाणो को ननजर राखवा केदु से अने बदाी सेडा न जिलो को कई भी संखा थाई तेर मेरे जाई नजर रखवा केदु है ।'
Decoded repr:  'बुडमण बंगाल नी पोलिस ने एक अगाड़ी नो पोलिस अधिकारी ने केदु की भारत अने बांगलादेश नी सेड़ा पर बदा धाणो को ननजर राखवा केदु से अने बदाी सेडा न जिलो को कई भी संखा थाई तेर मेरे जाई नजर रखवा केदु है ।'
Lengths:       192 -> 194
Pieces:        ['▁बुड', 'मण', '▁बंगाल', '▁नी', '▁पोलिस', '▁ने', '▁एक', '▁अगा', 'ड़ी', '

In [32]:
import unicodedata
import re

tokenizer_for_task = {
    "bhili": "Indic -> Indic 1B",
    "gondi": "Indic -> Indic 1B",
    "mundari": "Indic -> Indic 1B",
    "santali": "English -> Indic 1B",
}

def normalize_nfc(text):
    return unicodedata.normalize("NFC", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

for language, tokenizer_name in tokenizer_for_task.items():
    tok = tokenizer_results[tokenizer_name]
    _, target_col = language_columns[language]
    df = all_data[language]["validation"].head(100)

    exact = nfc_only = whitespace_only = both = unresolved = 0

    for original in df[target_col].astype(str):
        decoded = tok.tgt_spm.decode(
            tok.tgt_spm.encode(original, out_type=str)
        )

        if decoded == original:
            exact += 1
        elif normalize_nfc(decoded) == normalize_nfc(original):
            nfc_only += 1
        elif normalize_whitespace(decoded) == normalize_whitespace(original):
            whitespace_only += 1
        elif normalize_whitespace(normalize_nfc(decoded)) == normalize_whitespace(
            normalize_nfc(original)
        ):
            both += 1
        else:
            unresolved += 1

    print(f"\n{language.upper()}")
    print(f"Exact matches:                 {exact}")
    print(f"NFC resolves mismatch:         {nfc_only}")
    print(f"Whitespace resolves mismatch:  {whitespace_only}")
    print(f"Both NFC + whitespace resolve: {both}")
    print(f"Still different:               {unresolved}")


BHILI
Exact matches:                 84
NFC resolves mismatch:         4
Whitespace resolves mismatch:  8
Both NFC + whitespace resolve: 2
Still different:               2

GONDI
Exact matches:                 80
NFC resolves mismatch:         19
Whitespace resolves mismatch:  1
Both NFC + whitespace resolve: 0
Still different:               0

MUNDARI
Exact matches:                 53
NFC resolves mismatch:         39
Whitespace resolves mismatch:  4
Both NFC + whitespace resolve: 3
Still different:               1

SANTALI
Exact matches:                 97
NFC resolves mismatch:         0
Whitespace resolves mismatch:  3
Both NFC + whitespace resolve: 0
Still different:               0


## Section 4.6 — Tokenizer Compatibility and Round-Trip Analysis

### 1. Objective

Before beginning model fine-tuning and vocabulary extension, we investigated the pretrained IndicTrans2 tokenizers to understand their vocabulary structure, tokenization behavior on the four target languages, and ability to preserve text through SentencePiece encoding and decoding.

The target languages are Bhili, Gondi, Mundari, and Santali. The translation directions are Hindi → Bhili, Hindi → Gondi, Hindi → Mundari, and English → Santali.

### 2. Tokenizer and Vocabulary Findings

All five candidate checkpoints use the `IndicTransTokenizer` class. Each tokenizer exposes separate source and target SentencePiece models.

| Candidate checkpoint | Source vocabulary | Target vocabulary |
|---|---:|---:|
| Indic → English 1B | 128k | 32k |
| English → Indic 1B | 32k | 128k |
| Indic → Indic 1B | 128k | 128k |
| English → Indic distilled 200M | 32k | 128k |
| Indic → Indic distilled 320M | 128k | 128k |

The 32k and 128k SentencePiece vocabularies have different fingerprints, confirming that they are distinct vocabulary configurations.

The tokenizer special-token IDs are consistent across the inspected checkpoints: BOS = 0, PAD = 1, EOS = 2, and UNK = 3.

### 3. Tokenization Fertility Analysis

A diagnostic sample of 2,000 target sentences per language was used to compare tokenization with the 32k and 128k SentencePiece vocabularies.

| Language | 32k tokens/sentence | 128k tokens/sentence | 32k tokens/word | 128k tokens/word |
|---|---:|---:|---:|---:|
| Bhili | 105.506 | 30.831 | 4.957 | 1.448 |
| Gondi | 74.555 | 23.679 | 5.532 | 1.757 |
| Mundari | 78.115 | 30.438 | 5.530 | 2.155 |
| Santali | 39.544 | 28.102 | 2.029 | 1.442 |

The 128k vocabulary produced fewer tokens per sentence and per word for all four target languages in this sample. The measured split-token rates also decreased:

| Language | 32k split-token rate | 128k split-token rate |
|---|---:|---:|
| Bhili | 98.264% | 32.350% |
| Gondi | 99.547% | 48.418% |
| Mundari | 97.795% | 62.959% |
| Santali | 99.574% | 27.239% |

These results indicate differences in tokenization efficiency. They do **not** establish which checkpoint will produce better translations, and they do not by themselves prove that vocabulary extension will improve translation quality.

### 4. Language Support and Preprocessing

The inspected IndicTrans2 model-card language lists include Santali with the code `sat_Olck`, referring to Santali in the Ol Chiki script. Bhili, Gondi, and Mundari were not listed among the officially supported languages in the materials examined.

The installed `IndicProcessor` accepts parameters for source and target language codes and can preprocess an English-to-Santali example. The output includes the language-code prefix:

`eng_Latn sat_Olck`

A Hindi-source preprocessing example also succeeded with the `hin_Deva` source code.

The processor implementation is compiled, so direct source inspection using Python's `inspect.getsource()` was not available. The observed preprocessing behavior establishes that the tested calls work; it does not establish that unsupported languages are natively supported by the pretrained model.

### 5. SentencePiece Round-Trip Experiment

For each language, 100 validation targets were encoded using the task-specific checkpoint's target SentencePiece model and then decoded. The decoded strings were compared with the originals using exact string equality.

| Language | Exact matches | Exact-match rate | Initial mismatches |
|---|---:|---:|---:|
| Bhili | 84/100 | 84% | 16 |
| Gondi | 80/100 | 80% | 20 |
| Mundari | 53/100 | 53% | 47 |
| Santali | 97/100 | 97% | 3 |

The initial results showed substantial differences in exact round-trip preservation, especially for Mundari.

### 6. Investigation of Round-Trip Mismatches

Sample inspection revealed several types of differences:

- **Unicode normalization:** Characters that look similar may have different Unicode representations. For example, `अगाड़ी` may decode as `अगाड़ी`.
- **Whitespace normalization:** Repeated spaces may be collapsed into a single space.
- **Zero-width characters:** Some original strings contain invisible characters that may not survive the round trip.
- **Other differences:** A small number of examples remain different even after the normalization checks.

To investigate further, the original and decoded strings were compared using NFC Unicode normalization and whitespace normalization. Each example was assigned to the first category that resolved its mismatch.

| Language | Exact | NFC resolves | Whitespace resolves | Both resolve | Still different |
|---|---:|---:|---:|---:|---:|
| Bhili | 84 | 4 | 8 | 2 | 2 |
| Gondi | 80 | 19 | 1 | 0 | 0 |
| Mundari | 53 | 39 | 4 | 3 | 1 |
| Santali | 97 | 0 | 3 | 0 | 0 |

For this diagnostic sample:

- All 20 Gondi mismatches were resolved by NFC or whitespace normalization.
- 46 of 47 Mundari mismatches were resolved by the tested normalization checks.
- 14 of 16 Bhili mismatches were resolved.
- All three Santali mismatches were resolved by whitespace normalization.

These checks are diagnostic approximations. Because the categories are evaluated in order, an example resolved by NFC is not subsequently counted as a whitespace-resolved example. The remaining differences require further investigation.

### 7. Interpretation and Implications

The round-trip results show that exact string equality alone can underestimate text preservation when a tokenizer normalizes Unicode or whitespace. Consequently, the initial exact-match rates should not be interpreted as direct measures of vocabulary coverage or translation quality.

For subsequent experiments:

1. Preserve the original train, validation, and test data.
2. Do not silently normalize the corpus solely to improve round-trip scores.
3. Document the text-normalization behavior of each tokenizer.
4. Investigate the remaining unexplained mismatches before drawing conclusions about information loss.
5. Measure vocabulary-extension benefits using held-out translation quality and tokenization efficiency, rather than round-trip equality alone.

### 8. Provisional Conclusion

The tokenizer investigation establishes three useful findings. First, the inspected checkpoints use distinct 32k and 128k SentencePiece vocabularies. Second, the 128k vocabulary produces fewer tokens per word in the sampled target texts. Third, many observed round-trip mismatches are explained by Unicode or whitespace normalization.

These findings motivate further investigation of vocabulary-aware adaptation. However, the language-conditioning setup for Bhili, Gondi, and Mundari still needs to be established before selecting the baseline training procedure. No conclusion about translation quality or the effectiveness of vocabulary extension can yet be drawn from these tokenizer diagnostics.

In [33]:
# Section 4.7 — Inspect language-code preprocessing behavior

from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

source_example = ["This is a short translation test."]

tests = [
    ("Known English -> Santali", "eng_Latn", "sat_Olck"),
    ("Unknown target-code probe", "eng_Latn", "xyz_Test"),
]

for label, src_lang, tgt_lang in tests:
    print(f"\n{'=' * 60}")
    print(label)
    print(f"Source code: {src_lang} | Target code: {tgt_lang}")

    try:
        result = ip.preprocess_batch(
            source_example,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )
        print("Preprocessing accepted.")
        print("Output:", result)
    except Exception as e:
        print("Preprocessing rejected.")
        print("Error type:", type(e).__name__)
        print("Error:", str(e)[:500])


Known English -> Santali
Source code: eng_Latn | Target code: sat_Olck
Preprocessing accepted.
Output: ['eng_Latn sat_Olck This is a short translation test .']

Unknown target-code probe
Source code: eng_Latn | Target code: xyz_Test
Preprocessing accepted.
Output: ['eng_Latn xyz_Test This is a short translation test .']


## 5. Language Conditioning and Model Compatibility

### Objective

Before training the baseline LoRA model, we must establish how the pretrained IndicTrans2 model receives source- and target-language information and whether its existing language-conditioning mechanism is appropriate for our four translation tasks.

Our tasks are:

- Hindi → Bhili
- Hindi → Gondi
- Hindi → Mundari
- English → Santali

The previous experiments established that `IndicProcessor` accepts both known and arbitrary language-code strings. However, accepting a code during preprocessing does not demonstrate that the pretrained model supports the corresponding language.

We will therefore inspect the model's actual tokenization, embedding, and generation interfaces before choosing a conditioning strategy.

### Research principle

The baseline and vocabulary-extended experiments must use the same language-conditioning convention. Otherwise, any observed translation-quality difference could result from changing the conditioning strategy rather than extending the vocabulary.

We will preserve the existing dataset splits and original text. Any additional language codes or conditioning conventions must be documented explicitly and applied consistently during training, validation, and inference.

### Expected outcome

Identify a defensible conditioning strategy for each translation task, document its limitations, and verify that the model can process the resulting inputs before starting baseline LoRA training.

In [34]:

# Section 5.2: Inspect language-conditioning interfaces
# This is a diagnostic only; no model training or dataset modification.

import inspect
from transformers import AutoConfig, AutoTokenizer

MODEL_TO_INSPECT = "ai4bharat/indictrans2-indic-indic-1B"

print("Model:", MODEL_TO_INSPECT)

# Inspect the configuration without loading the model weights.
config = AutoConfig.from_pretrained(
    MODEL_TO_INSPECT,
    trust_remote_code=True,
    token=hf_token
)

print("\n--- Model configuration ---")
for key in [
    "model_type",
    "architectures",
    "vocab_size",
    "decoder_start_token_id",
    "bos_token_id",
    "eos_token_id",
    "pad_token_id",
    "tie_word_embeddings",
]:
    print(f"{key}: {getattr(config, key, None)}")

# Reuse the existing tokenizer result if it is already loaded.
# Otherwise, load the tokenizer for this checkpoint.
print("\n--- Tokenizer class and special tokens ---")
tok = AutoTokenizer.from_pretrained(
    MODEL_TO_INSPECT,
    trust_remote_code=True,
    token=hf_token
)

print("Tokenizer class:", type(tok).__name__)
print("Tokenizer length:", len(tok))
print("Special tokens map:", getattr(tok, "special_tokens_map", None))
print("Added vocabulary size:", len(tok.get_added_vocab()))

# Check the model-facing tokenizer methods and available language-related fields.
print("\n--- Tokenizer attributes related to language ---")
for attr in dir(tok):
    if any(term in attr.lower() for term in [
        "lang", "src", "tgt", "prefix", "special", "spm"
    ]):
        try:
            value = getattr(tok, attr)
            if not callable(value):
                print(f"{attr}: {str(value)[:300]}")
        except Exception as exc:
            print(f"{attr}: <could not inspect: {type(exc).__name__}>")

print("\n--- Tokenizer call signature ---")
try:
    print(inspect.signature(tok.__call__))
except Exception as exc:
    print("Could not inspect:", repr(exc))

print("\n--- Tokenizer class methods mentioning language or prefix ---")
for name in dir(type(tok)):
    if any(term in name.lower() for term in ["lang", "prefix", "prepare", "encode"]):
        method = getattr(type(tok), name, None)
        if callable(method):
            try:
                print(f"{name}{inspect.signature(method)}")
            except Exception:
                print(name)


Model: ai4bharat/indictrans2-indic-indic-1B

--- Model configuration ---
model_type: IndicTrans
architectures: ['IndicTransForConditionalGeneration']
vocab_size: 122672
decoder_start_token_id: 2
bos_token_id: 0
eos_token_id: 2
pad_token_id: 1
tie_word_embeddings: True

--- Tokenizer class and special tokens ---
Tokenizer class: IndicTransTokenizer
Tokenizer length: 122706
Special tokens map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
Added vocabulary size: 4

--- Tokenizer attributes related to language ---
SPECIAL_TOKENS_ATTRIBUTES: ['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']
_additional_special_tokens: []
additional_special_tokens: []
additional_special_tokens_ids: []
all_special_ids: [0, 2, 3, 1]
all_special_tokens: ['<s>', '</s>', '<unk>', '<pad>']
all_special_tokens_extended: [AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, spe

In [35]:

# Section 5.3: Inspect the language-tag extension method
# Diagnostic only — no vocabulary or model weights are modified.

import inspect

tok_class = type(tok)

print("Tokenizer class:", tok_class)
print("\n--- add_new_language_tags source ---")

try:
    print(inspect.getsource(tok_class.add_new_language_tags))
except (OSError, TypeError) as exc:
    print("Source unavailable:", repr(exc))

print("\n--- Relevant tokenizer attributes ---")
for attr in [
    "src_vocab_size",
    "tgt_vocab_size",
    "src_encoder",
    "tgt_encoder",
    "src_decoder",
    "tgt_decoder",
    "src_spm",
    "tgt_spm",
]:
    value = getattr(tok, attr, None)

    if isinstance(value, dict):
        print(f"{attr}: dictionary with {len(value)} entries")
    elif value is not None:
        print(f"{attr}: {type(value).__name__} = {value}")
    else:
        print(f"{attr}: not available")

print("\n--- Language-tag examples already in the vocabularies ---")
for tag in [
    "eng_Latn",
    "hin_Deva",
    "sat_Olck",
    "bhil",
    "gon",
    "unrecognized_Test",
]:
    print(
        f"{tag!r}: "
        f"source_id={tok.src_encoder.get(tag)}, "
        f"target_id={tok.tgt_encoder.get(tag)}"
    )


Tokenizer class: <class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans.IndicTransTokenizer'>

--- add_new_language_tags source ---
    def add_new_language_tags(self, new_tags: List[str]) -> None:
        global LANGUAGE_TAGS
        LANGUAGE_TAGS = frozenset(LANGUAGE_TAGS | set(new_tags))


--- Relevant tokenizer attributes ---
src_vocab_size: int = 122706
tgt_vocab_size: int = 122672
src_encoder: dictionary with 122706 entries
tgt_encoder: dictionary with 122672 entries
src_decoder: dictionary with 122706 entries
tgt_decoder: dictionary with 122672 entries
src_spm: SentencePieceProcessor = <sentencepiece.SentencePieceProcessor object at 0x7c4286df9850>
tgt_spm: SentencePieceProcessor = <sentencepiece.SentencePieceProcessor object at 0x7c42871bbee0>

--- Language-tag examples already in the vocabularies ---
'eng_Latn': source_id=4, target_id=None
'hin_Deva': source_id=8, target_id=None
'sat_Olck': source_id=2

### 5.5. Pilot Selection and Experimental Scope

We will first investigate two representative cases:

1. English → Santali, using the supported `eng_Latn` and `sat_Olck` language tags.
2. Hindi → Bhili, using `hin_Deva` and the documented `mar_Deva` proxy.

These pilots serve different purposes: Santali provides a supported-target control, while Bhili tests a proxy-based adaptation strategy.

Successful preprocessing or tokenization will not be treated as proof of translation capability. We will verify the actual model inputs and target-label construction before training.

After validating the baseline pipeline, we will extend the experiments to Gondi and Mundari. Proxy selection based on linguistic similarity and transfer learning will be treated as a separate, controlled investigation.

The primary vocabulary-extension experiment will keep the language-conditioning strategy fixed between baseline and modified models.

In [36]:

# Section 5.6: Inspect the two pilot input formats
# This only preprocesses sample sentences; it does not train a model.

from IndicTransToolkit import IndicProcessor

pilot_processor = IndicProcessor(inference=True)

pilot_configs = {
    "Santali": {
        "source_lang": "eng_Latn",
        "target_lang": "sat_Olck",
        "data_key": "santali",
        "source_column": "English",
        "target_column": "Santali",
    },
    "Bhili": {
        "source_lang": "hin_Deva",
        "target_lang": "mar_Deva",
        "data_key": "bhili",
        "source_column": "Hindi",
        "target_column": "Bhili",
    },
}

for name, cfg in pilot_configs.items():
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    row = all_data[cfg["data_key"]]["train"].iloc[0]
    source_text = str(row[cfg["source_column"]])
    target_text = str(row[cfg["target_column"]])

    processed_source = pilot_processor.preprocess_batch(
        [source_text],
        src_lang=cfg["source_lang"],
        tgt_lang=cfg["target_lang"],
    )

    print("Source language tag:", cfg["source_lang"])
    print("Target/proxy tag:", cfg["target_lang"])
    print("Original source:", repr(source_text))
    print("Original target:", repr(target_text))
    print("Processed source:", repr(processed_source[0]))



==================== Santali ====================
Source language tag: eng_Latn
Target/proxy tag: sat_Olck
Original source: "Instacart's Shopper app looks like a green Instacart bag on a white background."
Original target: 'ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾'
Processed source: "eng_Latn sat_Olck Instacart 's Shopper app looks like a green Instacart bag on a white background ."

==================== Bhili ====================
Source language tag: hin_Deva
Target/proxy tag: mar_Deva
Original source: 'युवा उद्यमी, जो किसी स्टार्टअप की योजना बना रहे हों, वे समग्र स्वास्थ्य सेवाओं में कई अवसर प्राप्त कर सकते हैं।'
Original target: 'खुद नो नीजी काम काज करवा वाळा जोवान मनखा ,ने कईनु काम सलाडवा नी ओजना बणाविरा से।, तिहुंनें हगळी स्वास्थ्य से।वायु मां घणा मौका मळी सके से। ।'
Processed source: 'hin_Deva mar_Deva युवा उद्यमी , जो किसी स्टार्टअप की योजना बना रहे हों , वे समग्र स्वास्थ्य सेवाओं में कई अवसर प्राप्त कर सकते हैं ।'


In [37]:

# Section 5.7: Inspect source prefixes and target labels
# Diagnostic only — no model training or tokenizer modification.

from transformers import AutoTokenizer

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"

check_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)

for name, cfg in pilot_configs.items():
    print(f"\n{'=' * 18} {name} {'=' * 18}")

    row = all_data[cfg["data_key"]]["train"].iloc[0]

    source_text = str(row[cfg["source_column"]])
    target_text = str(row[cfg["target_column"]])

    processed_source = pilot_processor.preprocess_batch(
        [source_text],
        src_lang=cfg["source_lang"],
        tgt_lang=cfg["target_lang"],
    )[0]

    # Encode the processed source and the unmodified target independently.
    source_ids = check_tokenizer(
        processed_source,
        add_special_tokens=True
    )["input_ids"]

    target_ids = check_tokenizer(
        text_target=target_text,
        add_special_tokens=True
    )["input_ids"]

    print("Source prefix:", processed_source.split()[:2])
    print("Source first token IDs:", source_ids[:12])
    print("Source first tokens:", check_tokenizer.convert_ids_to_tokens(source_ids[:12]))

    print("\nTarget original text:", repr(target_text[:160]))
    print("Target first token IDs:", target_ids[:12])
    print("Target first tokens:", check_tokenizer.convert_ids_to_tokens(target_ids[:12]))

    print("\nSource ends with EOS:", source_ids[-1] == check_tokenizer.eos_token_id)
    print("Target ends with EOS:", target_ids[-1] == check_tokenizer.eos_token_id)



================== Santali ==================
Source prefix: ['eng_Latn', 'sat_Olck']
Source first token IDs: [4, 29925, 57861, 18782, 38095, 40, 1888, 59633, 30179, 42833, 91069, 51927]
Source first tokens: ['eng_Latn', 'sat_Olck', '▁Inst', 'ac', 'art', "▁'", 's', '▁Shop', 'per', '▁app', '▁looks', '▁like']

Target original text: 'ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾'
Target first token IDs: [95037, 72860, 90936, 95244, 78321, 97232, 102588, 103847, 78143, 98715, 48377, 103575]
Target first tokens: ['▁مروج', '▁وٲل', '▁अशोय़', '▁सुनै', '▁गाहाइ', '▁आन्जाद', 'ꯔꯕꯤ', '▁थगाय', '▁घुरि', '▁देरहानो', '▁وڪرو', '▁दुवार']

Source ends with EOS: True
Target ends with EOS: True

================== Bhili ==================
Source prefix: ['hin_Deva', 'mar_Deva']
Source first token IDs: [8, 18, 4614, 40975, 6, 99, 432, 31817, 21, 1020, 2245, 169]
Source first tokens: ['hin_Deva', 'mar_Deva', '▁युवा', '▁उद्यमी', '▁,', '▁जो', '▁किसी', '▁स्टार्टअप', '▁की', '▁योजना'

In [38]:

# Section 5.8: Compare target-side tokenization across checkpoints
# Diagnostic only — does not train or modify model weights.

from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

# Use the actual first Santali and Bhili training examples.
samples = {
    "Santali": str(all_data["santali"]["train"].iloc[0]["Santali"]),
    "Bhili": str(all_data["bhili"]["train"].iloc[0]["Bhili"]),
}

for model_label, model_id in TOKENIZER_MODELS.items():
    print(f"\n{'=' * 18} {model_label} {'=' * 18}")

    current_tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    print("Tokenizer:", type(current_tok).__name__)

    for language, text in samples.items():
        ids = current_tok(
            text_target=text,
            add_special_tokens=True,
        )["input_ids"]

        pieces = current_tok.convert_ids_to_tokens(ids[:20])
        decoded = current_tok.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        print(f"\n{language}")
        print("Original:", repr(text[:140]))
        print("Token pieces:", pieces)
        print("Decoded:", repr(decoded[:180]))
        print("Exact round trip:", decoded == text)



================== Indic -> Indic 1B ==================
Tokenizer: IndicTransTokenizer

Santali
Original: 'ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾'
Token pieces: ['▁مروج', '▁وٲل', '▁अशोय़', '▁सुनै', '▁गाहाइ', '▁आन्जाद', 'ꯔꯕꯤ', '▁थगाय', '▁घुरि', '▁देरहानो', '▁وڪرو', '▁दुवार', 'ئڪ', 'लाय़', '▁mur', '▁ہڙتال', 'ጵ', '▁مروج', '▁وٲل', '▁अशोय़']
Decoded: 'ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾'
Exact round trip: True

Bhili
Original: 'खुद नो नीजी काम काज करवा वाळा जोवान मनखा ,ने कईनु काम सलाडवा नी ओजना बणाविरा से।, तिहुंनें हगळी स्वास्थ्य से।वायु मां घणा मौका मळी सके से। ।'
Token pieces: ['▁कट्ट', '▁केली', '▁इत्', '▁नल्ल', 'नु', '▁अपनी', 'दे', '▁चेस्तुन्नारनि', 'ं', '▁हेउछि', 'त्तिल्', '▁अड', '▁.', 'ৰ', 'ित', '▁एकटि', 'नु', 'सिऩ्', '▁रेजिस्ट', '▁इत्']
Decoded: 'खुद नो नीजी काम काज करवा वाळा जोवान मनखा ,ने कईनु काम सलाडवा नी ओजना बणाविरा से।, तिहुंनें हगळी स्वास्थ्य से।वायु मां घणा मौका मळी सके से। ।'
Exact round trip: Tr

In [39]:

# Section 5.9: Measure target tokenization quality
# No model weights are loaded or changed.

from transformers import AutoTokenizer
import numpy as np

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

TARGET_COLUMNS = {
    "bhili": "Bhili",
    "gondi": "Gondi",
    "mundari": "Mundari",
    "santali": "Santali",
}

results = []

for model_label, model_id in TOKENIZER_MODELS.items():
    print(f"\nLoading {model_label}...")
    current_tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    unk_id = current_tok.unk_token_id

    for language, target_col in TARGET_COLUMNS.items():
        texts = (
            all_data[language]["validation"][target_col]
            .dropna()
            .astype(str)
            .head(100)
            .tolist()
        )

        unk_count = 0
        total_tokens = 0
        affected_sentences = 0
        roundtrip_exact = 0

        for text in texts:
            ids = current_tok(
                text_target=text,
                add_special_tokens=False,
            )["input_ids"]

            sentence_unk_count = sum(token_id == unk_id for token_id in ids)
            unk_count += sentence_unk_count
            total_tokens += len(ids)

            if sentence_unk_count:
                affected_sentences += 1

            decoded = current_tok.decode(
                ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            if decoded == text:
                roundtrip_exact += 1

        result = {
            "model": model_label,
            "language": language,
            "examples": len(texts),
            "unknown_tokens": unk_count,
            "total_tokens": total_tokens,
            "unknown_token_rate_pct": (
                100 * unk_count / total_tokens if total_tokens else 0
            ),
            "sentences_with_unknown_pct": (
                100 * affected_sentences / len(texts) if texts else 0
            ),
            "exact_roundtrip_pct": (
                100 * roundtrip_exact / len(texts) if texts else 0
            ),
        }
        results.append(result)

        print(
            f"{language:8} | "
            f"UNK={unk_count:6} | "
            f"tokens={total_tokens:7} | "
            f"UNK rate={result['unknown_token_rate_pct']:7.3f}% | "
            f"sentences affected={result['sentences_with_unknown_pct']:6.1f}% | "
            f"exact round-trip={result['exact_roundtrip_pct']:6.1f}%"
        )

import pandas as pd
tokenization_report = pd.DataFrame(results)
display(tokenization_report)



Loading Indic -> Indic 1B...
bhili    | UNK=     2 | tokens=   3408 | UNK rate=  0.059% | sentences affected=   2.0% | exact round-trip=  83.0%
gondi    | UNK=     4 | tokens=   2397 | UNK rate=  0.167% | sentences affected=   2.0% | exact round-trip=  78.0%
mundari  | UNK=    21 | tokens=   2812 | UNK rate=  0.747% | sentences affected=  14.0% | exact round-trip=  46.0%
santali  | UNK=    54 | tokens=   2723 | UNK rate=  1.983% | sentences affected=  29.0% | exact round-trip=  69.0%

Loading English -> Indic 1B...
bhili    | UNK=     2 | tokens=   3408 | UNK rate=  0.059% | sentences affected=   2.0% | exact round-trip=  83.0%
gondi    | UNK=     4 | tokens=   2397 | UNK rate=  0.167% | sentences affected=   2.0% | exact round-trip=  78.0%
mundari  | UNK=    21 | tokens=   2812 | UNK rate=  0.747% | sentences affected=  14.0% | exact round-trip=  46.0%
santali  | UNK=    54 | tokens=   2723 | UNK rate=  1.983% | sentences affected=  29.0% | exact round-trip=  69.0%


,model,language,examples,unknown_tokens,total_tokens,unknown_token_rate_pct,sentences_with_unknown_pct,exact_roundtrip_pct
0,Indic -> Indic 1B,bhili,100,2,3408,0.058685,2.0,83.0
1,Indic -> Indic 1B,gondi,100,4,2397,0.166875,2.0,78.0
2,Indic -> Indic 1B,mundari,100,21,2812,0.746799,14.0,46.0
3,Indic -> Indic 1B,santali,100,54,2723,1.983107,29.0,69.0
4,English -> Indic 1B,bhili,100,2,3408,0.058685,2.0,83.0
5,English -> Indic 1B,gondi,100,4,2397,0.166875,2.0,78.0
6,English -> Indic 1B,mundari,100,21,2812,0.746799,14.0,46.0
7,English -> Indic 1B,santali,100,54,2723,1.983107,29.0,69.0


In [40]:
from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

for label, model_id in TOKENIZER_MODELS.items():
    print(f"\n{'=' * 65}")
    print(label)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token
    )

    print("Tokenizer class:", type(tok).__name__)
    print("Tokenizer length:", len(tok))
    print("UNK token ID:", tok.unk_token_id)

    for attr in [
        "src_vocab_size",
        "tgt_vocab_size",
        "src_encoder",
        "tgt_encoder",
        "src_spm",
        "tgt_spm",
    ]:
        obj = getattr(tok, attr, None)

        if obj is None:
            print(f"{attr}: unavailable")
        elif isinstance(obj, dict):
            print(f"{attr}: {len(obj)} entries")
        elif isinstance(obj, (int, str)):
            print(f"{attr}: {obj}")
        else:
            print(f"{attr}: {type(obj).__name__}")

    print("Target tokenization example:")
    example = all_data["santali"]["validation"]["Santali"].iloc[0]
    encoded = tok(text_target=str(example), add_special_tokens=False)
    ids = encoded["input_ids"]

    print("Target text:", repr(str(example)[:150]))
    print("First 20 IDs:", ids[:20])
    print("First 20 token strings:", tok.convert_ids_to_tokens(ids[:20]))
    print("UNK IDs in target encoding:", sum(i == tok.unk_token_id for i in ids))


Indic -> Indic 1B
Tokenizer class: IndicTransTokenizer
Tokenizer length: 122706
UNK token ID: 3
src_vocab_size: 122706
tgt_vocab_size: 122672
src_encoder: 122706 entries
tgt_encoder: 122672 entries
src_spm: SentencePieceProcessor
tgt_spm: SentencePieceProcessor
Target tokenization example:
Target text: 'ᱢᱤᱱᱦᱟᱡ ᱟᱜ ᱠᱟᱛᱷᱟᱛᱮ, ᱤᱞᱛᱩᱫᱢᱤᱥ ᱟᱹᱰᱤᱫᱷᱟᱣ ᱥᱟᱸᱜᱷᱟᱨ ᱨᱮ ᱵᱟᱫᱟᱩᱱ, ᱟᱣᱭᱟᱫᱷ, ᱵᱮᱱᱟᱨᱚᱥ ᱟᱨ ᱥᱤᱵᱟᱞᱤᱠ ᱪᱮᱛᱟᱱ ᱫᱚᱲᱦᱟᱛᱮ ᱫᱤᱞᱞᱤ ᱜᱚᱵᱚᱞᱨᱮ ᱛᱷᱟᱯᱚᱱ ᱞᱮᱫᱟᱭ ᱾'
First 20 IDs: [110836, 87980, 79698, 74441, 89703, 66424, 94120, 103699, 100590, 74321, 108212, 108073, 87627, 100367, 96137, 41991, 88966, 84920, 83986, 94120]
First 20 token strings: ['▁ᱛᱷᱟᱢ', '▁dam', 'धेनु', '▁میلو', '▁مصی', '▁local', '▁وڻج', '▁तोड़ि', '▁ꯁꯤꯡꯍꯅ', '▁نظارو', 'ाजबंशी', '▁मेलखौ', '▁bike', '▁ꯑꯟ', '▁آھ', '▁مسٹر', '▁Krish', '▁اہن', '▁تسبیح', '▁وڻج']
UNK IDs in target encoding: 0

English -> Indic 1B
Tokenizer class: IndicTransTokenizer
Tokenizer length: 32322
UNK token ID: 3
src_vocab_size: 32322
tgt_vocab_size: 122672
src_encoder: 32322 entries
tgt

In [41]:
from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

example = str(
    all_data["santali"]["validation"]["Santali"].iloc[0]
)

for label, model_id in TOKENIZER_MODELS.items():
    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    print(f"\n{'=' * 60}")
    print(label)

    # Inspect the target SentencePiece model directly.
    sp = tok.tgt_spm
    pieces = sp.encode(example, out_type=str)
    ids = sp.encode(example, out_type=int)
    decoded = sp.decode(ids)

    print("Target SentencePiece vocabulary:", sp.get_piece_size())
    print("Target piece count:", len(pieces))
    print("Target UNK piece count:", sum(p == "<unk>" for p in pieces))
    print("Target UNK ID count:", sum(i == sp.unk_id() for i in ids))
    print("First 20 target pieces:", pieces[:20])
    print("SentencePiece round-trip exact:", decoded == example)
    print("SentencePiece round-trip NFC:",
          __import__("unicodedata").normalize("NFC", decoded)
          == __import__("unicodedata").normalize("NFC", example))


Indic -> Indic 1B
Target SentencePiece vocabulary: 128000
Target piece count: 39
Target UNK piece count: 0
Target UNK ID count: 0
First 20 target pieces: ['▁ᱢᱤᱱ', 'ᱦᱟ', 'ᱡ', '▁ᱟᱜ', '▁ᱠᱟᱛᱷᱟ', 'ᱛᱮ', ',', '▁ᱤᱞ', 'ᱛᱩ', 'ᱫ', 'ᱢᱤᱥ', '▁ᱟᱹᱰ', 'ᱤᱫ', 'ᱷᱟᱣ', '▁ᱥᱟᱸᱜᱷᱟᱨ', '▁ᱨᱮ', '▁ᱵᱟ', 'ᱫᱟ', 'ᱩᱱ', ',']
SentencePiece round-trip exact: True
SentencePiece round-trip NFC: True

English -> Indic 1B
Target SentencePiece vocabulary: 128000
Target piece count: 39
Target UNK piece count: 0
Target UNK ID count: 0
First 20 target pieces: ['▁ᱢᱤᱱ', 'ᱦᱟ', 'ᱡ', '▁ᱟᱜ', '▁ᱠᱟᱛᱷᱟ', 'ᱛᱮ', ',', '▁ᱤᱞ', 'ᱛᱩ', 'ᱫ', 'ᱢᱤᱥ', '▁ᱟᱹᱰ', 'ᱤᱫ', 'ᱷᱟᱣ', '▁ᱥᱟᱸᱜᱷᱟᱨ', '▁ᱨᱮ', '▁ᱵᱟ', 'ᱫᱟ', 'ᱩᱱ', ',']
SentencePiece round-trip exact: True
SentencePiece round-trip NFC: True



## 5.10 Model Architecture and Embedding Compatibility Verification

### Objective

Before beginning baseline LoRA training or vocabulary extension, we need to understand how the pretrained IndicTrans2 model represents source and target tokens internally.

The tokenizer inspection in Section 5.9 established that the model has separate source and target vocabularies. For the Indic-to-Indic 1B checkpoint, the tokenizer reports 122,706 source vocabulary entries and 122,672 target vocabulary entries, while the model configuration reports a vocabulary size of 122,672.

These differences make it important to inspect the actual model architecture instead of assuming that the configuration's `vocab_size` describes every embedding matrix.

### What We Will Investigate

1. **Model configuration:** Identify the model architecture, configured vocabulary size, and weight-tying setting.
2. **Encoder embeddings:** Determine the number of embedding rows available for source token IDs.
3. **Decoder embeddings:** Determine the number of embedding rows available for target token IDs.
4. **Output embeddings:** Check the dimensions of the output projection and its compatibility with target token IDs.
5. **Vocabulary compatibility:** Verify that the token IDs produced by the tokenizer can be represented by the corresponding model components.

### Why This Matters for Our Research

Our primary experiment compares the original tokenizer and vocabulary against a vocabulary-extended configuration. To make this comparison meaningful, both experimental settings must preserve a valid mapping between token IDs and model parameters.

If vocabulary extension is implemented incorrectly, token IDs may exceed embedding dimensions, pretrained representations may become misaligned, or existing model parameters may be unintentionally changed.

Therefore, we will first establish the original model's embedding structure. We will not resize embeddings or modify vocabulary mappings until their compatibility has been verified.

### Expected Outcome

A clear record of the model's encoder, decoder, and output embedding dimensions, together with a verified mapping between source and target token IDs and their corresponding model components.

This will provide the architectural baseline needed before proceeding to LoRA training and the vocabulary-extension experiment.


In [42]:

import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("Model architecture:", config.architectures)
print("Model type:", config.model_type)
print("Configured vocabulary size:", config.vocab_size)
print("Tied word embeddings:", config.tie_word_embeddings)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

def print_embedding_shape(label, module):
    if module is None:
        print(f"{label}: unavailable")
        return

    weight = getattr(module, "weight", None)
    print(
        f"{label}:",
        tuple(weight.shape) if weight is not None else "no weight matrix"
    )

print("\n--- Embedding matrices ---")

print_embedding_shape(
    "Input embeddings",
    model.get_input_embeddings()
)

print_embedding_shape(
    "Output embeddings",
    model.get_output_embeddings()
)

for component_name in ("encoder", "decoder"):
    component = getattr(model, component_name, None)

    if component is None and hasattr(model, "model"):
        component = getattr(model.model, component_name, None)

    print(f"\n{component_name.capitalize()} component:")

    if component is not None:
        print_embedding_shape(
            f"{component_name} token embeddings",
            getattr(component, "embed_tokens", None)
        )
    else:
        print("Component not found at the expected attribute path.")

print("\n--- Memory cleanup ---")
del model
torch.cuda.empty_cache()


Model architecture: ['IndicTransForConditionalGeneration']
Model type: IndicTrans
Configured vocabulary size: 122672
Tied word embeddings: True


modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]


--- Embedding matrices ---
Input embeddings: (122706, 1024)
Output embeddings: (122672, 1024)

Encoder component:
encoder token embeddings: (122706, 1024)

Decoder component:
decoder token embeddings: (122672, 1024)

--- Memory cleanup ---



### 5.10.2 Results and Interpretation

The model architecture and embedding matrices were inspected using the `ai4bharat/indictrans2-indic-indic-1B` checkpoint.

#### Observed Results

| Component | Embedding shape | Vocabulary entries |
|---|---:|---:|
| Encoder input embeddings | (122706, 1024) | 122,706 |
| Decoder token embeddings | (122672, 1024) | 122,672 |
| Output embeddings | (122672, 1024) | 122,672 |

Additional configuration details:

- Model architecture: `IndicTransForConditionalGeneration`
- Model type: `IndicTrans`
- Configured vocabulary size: `122672`
- `tie_word_embeddings`: `True`

#### Key Observations

**1. Source and target embedding dimensions differ in vocabulary size.**

The encoder has 122,706 embedding rows, whereas the decoder has 122,672. Both use an embedding dimension of 1,024.

The encoder therefore accommodates 34 more token IDs than the decoder. This is consistent with the tokenizer's previously observed difference between source and target vocabulary sizes.

**2. The configured vocabulary size does not describe both embedding matrices.**

Although the model configuration reports a vocabulary size of 122,672, the encoder embedding matrix contains 122,706 rows. Therefore, using `config.vocab_size` alone would not accurately describe the encoder's embedding capacity.

**3. Output embeddings match the decoder vocabulary size.**

The output embedding matrix has shape `(122672, 1024)`, matching the decoder token embedding dimensions. This is consistent with the model's configured vocabulary size.

**4. Weight tying requires further verification.**

The configuration reports `tie_word_embeddings=True`. This flag alone does not establish which parameters share weights in the instantiated architecture. The relationship between the decoder embeddings, output projection, and encoder embeddings should be checked directly if it affects the vocabulary-extension implementation.

#### Implications for Vocabulary Extension

These results establish the original embedding dimensions and explain why source and target vocabularies must be handled separately.

For our experiments, vocabulary extension must preserve the correspondence between source token IDs and encoder embeddings, as well as between target token IDs, decoder embeddings, and output logits.

We should not resize the model's embeddings yet. First, we must verify the actual token-ID ranges, special-token mappings, and parameter-sharing behavior. This will help prevent invalid IDs, incorrectly initialized embeddings, or unintended changes to pretrained parameters.

#### Conclusion

The inspected checkpoint has separate encoder and decoder embedding matrices with a common embedding dimension of 1,024 but different vocabulary sizes. This provides the architectural baseline for investigating vocabulary extension. The precise resizing and initialization strategy remains to be determined after the remaining compatibility checks.



### 5.10.3 Verification of Weight Tying and Parameter Sharing

#### Objective

The model configuration reports `tie_word_embeddings=True`. However, this setting alone does not tell us whether the encoder embeddings, decoder embeddings, and output projection actually share the same parameters.

We will inspect the instantiated model to determine:

1. Whether the encoder and decoder embedding matrices share the same underlying parameters.
2. Whether the decoder token embeddings and output projection share weights.
3. Whether the model's parameter shapes and storage relationships agree with the observed vocabulary dimensions.

#### Why This Matters

Vocabulary extension changes the number of token representations available to the model. If parameters are shared, resizing or replacing one embedding matrix may affect other components. If they are separate, each affected component may require its own compatible update.

Understanding these relationships is necessary to design a controlled vocabulary-extension experiment without unintentionally changing the pretrained model's architecture.

#### Method

We will compare parameter object identity, storage pointers, and tensor shapes. These checks help distinguish genuinely shared parameters from matrices that merely have matching dimensions.

No model weights will be modified in this section.


In [43]:

import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

encoder_embeddings = model.get_input_embeddings()
decoder_embeddings = model.get_decoder().embed_tokens
output_embeddings = model.get_output_embeddings()

def inspect_weight(label, module):
    weight = module.weight
    print(f"{label}:")
    print("  Shape:", tuple(weight.shape))
    print("  Parameter object ID:", id(weight))
    print("  Storage pointer:", weight.data_ptr())
    print("  Storage offset:", weight.storage_offset())

print("=== Embedding parameter inspection ===\n")

inspect_weight("Encoder embeddings", encoder_embeddings)
inspect_weight("Decoder embeddings", decoder_embeddings)
inspect_weight("Output embeddings", output_embeddings)

print("\n=== Parameter-sharing checks ===")

print(
    "Encoder and decoder share parameter object:",
    encoder_embeddings.weight is decoder_embeddings.weight
)

print(
    "Decoder and output share parameter object:",
    decoder_embeddings.weight is output_embeddings.weight
)

print(
    "Encoder and output share parameter object:",
    encoder_embeddings.weight is output_embeddings.weight
)

print("\n=== Shape compatibility ===")

print(
    "Decoder and output shapes match:",
    decoder_embeddings.weight.shape == output_embeddings.weight.shape
)

print(
    "Encoder and decoder shapes match:",
    encoder_embeddings.weight.shape == decoder_embeddings.weight.shape
)

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


=== Embedding parameter inspection ===

Encoder embeddings:
  Shape: (122706, 1024)
  Parameter object ID: 136624700288816
  Storage pointer: 136623045521472
  Storage offset: 0
Decoder embeddings:
  Shape: (122672, 1024)
  Parameter object ID: 136627967707568
  Storage pointer: 136623296827456
  Storage offset: 0
Output embeddings:
  Shape: (122672, 1024)
  Parameter object ID: 136627966590272
  Storage pointer: 136623799300160
  Storage offset: 0

=== Parameter-sharing checks ===
Encoder and decoder share parameter object: False
Decoder and output share parameter object: False
Encoder and output share parameter object: False

=== Shape compatibility ===
Decoder and output shapes match: True
Encoder and decoder shapes match: False


In [44]:

import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("Configuration tie_word_embeddings:",
      model.config.tie_word_embeddings)

print("\nDeclared tied-weight keys:")
print(getattr(model, "_tied_weights_keys", None))

print("\nDeclared keys to ignore on load:")
print(getattr(model, "_keys_to_ignore_on_load_missing", None))

def report_sharing(stage):
    encoder = model.get_input_embeddings()
    decoder = model.get_decoder().embed_tokens
    output = model.get_output_embeddings()

    print(f"\n--- {stage} ---")
    print("Encoder is decoder:", encoder.weight is decoder.weight)
    print("Decoder is output:", decoder.weight is output.weight)
    print("Encoder is output:", encoder.weight is output.weight)

report_sharing("Before built-in tie_weights()")

print("\nCalling model.tie_weights()...")
model.tie_weights()

report_sharing("After built-in tie_weights()")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Configuration tie_word_embeddings: True

Declared tied-weight keys:
['decoder.embed_tokens.weight', 'lm_head.weight']

Declared keys to ignore on load:
None

--- Before built-in tie_weights() ---
Encoder is decoder: False
Decoder is output: False
Encoder is output: False

Calling model.tie_weights()...

--- After built-in tie_weights() ---
Encoder is decoder: False
Decoder is output: False
Encoder is output: False


In [45]:

import inspect
import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

decoder_weight = model.get_decoder().embed_tokens.weight
output_weight = model.get_output_embeddings().weight

print("=== Parameter comparison ===")
print("Decoder shape:", tuple(decoder_weight.shape))
print("Output shape:", tuple(output_weight.shape))
print("Same object:", decoder_weight is output_weight)
print(
    "Same underlying storage:",
    decoder_weight.untyped_storage().data_ptr()
    == output_weight.untyped_storage().data_ptr()
)
print(
    "Exactly equal values:",
    torch.equal(decoder_weight, output_weight)
)

print("\n=== Model implementation ===")

for method_name in ("tie_weights", "get_output_embeddings"):
    method = getattr(type(model), method_name, None)
    print(f"\n{method_name}:")

    if method is None:
        print("Method not found on the model class.")
        continue

    try:
        print(inspect.getsource(method))
    except (OSError, TypeError) as exc:
        print("Source unavailable:", repr(exc))

print("\n=== Model class ===")
print(type(model))
print("Model source file:", inspect.getfile(type(model)))

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


=== Parameter comparison ===
Decoder shape: (122672, 1024)
Output shape: (122672, 1024)
Same object: False
Same underlying storage: False
Exactly equal values: False

=== Model implementation ===

tie_weights:
    def tie_weights(self):
        if self.config.share_decoder_input_output_embed:
            self._tie_or_clone_weights(self.model.decoder.embed_tokens, self.lm_head)


get_output_embeddings:
    def get_output_embeddings(self):
        return self.lm_head


=== Model class ===
<class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.modeling_indictrans.IndicTransForConditionalGeneration'>
Model source file: /root/.cache/huggingface/modules/transformers_modules/ai4bharat/indictrans2-indic-indic-1B/24d732922a0a91d0998d5568e3af37b7a21cd705/modeling_indictrans.py


In [46]:
from transformers import AutoConfig

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("tie_word_embeddings:",
      getattr(config, "tie_word_embeddings", None))

print("share_decoder_input_output_embed:",
      getattr(config, "share_decoder_input_output_embed", None))

print("\nAll config attributes related to embedding/tied weights:")

for key, value in config.to_dict().items():
    key_lower = key.lower()

    if (
        "embed" in key_lower
        or "tie" in key_lower
        or "share" in key_lower
        or "vocab" in key_lower
    ):
        print(f"{key}: {value}")

tie_word_embeddings: True
share_decoder_input_output_embed: False

All config attributes related to embedding/tied weights:
encoder_vocab_size: 122706
decoder_vocab_size: 122672
layernorm_embedding: False
encoder_embed_dim: 1024
decoder_embed_dim: 1024
scale_embedding: True
share_decoder_input_output_embed: False
tie_word_embeddings: True
tie_encoder_decoder: False
vocab_size: 122672


In [47]:
from transformers import AutoTokenizer

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("=== Vocabulary dimensions ===")
print("Tokenizer length:", len(tok))
print("Source vocabulary size:", tok.src_vocab_size)
print("Target vocabulary size:", tok.tgt_vocab_size)

print("\n=== Token ID range checks ===")

for name, vocab in [
    ("Source", tok.src_encoder),
    ("Target", tok.tgt_encoder),
]:
    ids = list(vocab.values())

    print(f"\n{name}:")
    print("Number of entries:", len(vocab))
    print("Minimum token ID:", min(ids))
    print("Maximum token ID:", max(ids))
    print("Unique token IDs:", len(set(ids)))
    print("Contiguous IDs:", set(ids) == set(range(max(ids) + 1)))

print("\n=== Special and language token IDs ===")

for token in [
    "<s>", "<pad>", "</s>", "<unk>",
    "eng_Latn", "hin_Deva", "mar_Deva", "sat_Olck",
]:
    print(
        repr(token),
        "| source ID:", tok.src_encoder.get(token),
        "| target ID:", tok.tgt_encoder.get(token),
    )

=== Vocabulary dimensions ===
Tokenizer length: 122706
Source vocabulary size: 122706
Target vocabulary size: 122672

=== Token ID range checks ===

Source:
Number of entries: 122706
Minimum token ID: 0
Maximum token ID: 122705
Unique token IDs: 122706
Contiguous IDs: True

Target:
Number of entries: 122672
Minimum token ID: 0
Maximum token ID: 122671
Unique token IDs: 122672
Contiguous IDs: True

=== Special and language token IDs ===
'<s>' | source ID: 0 | target ID: 0
'<pad>' | source ID: 1 | target ID: 1
'</s>' | source ID: 2 | target ID: 2
'<unk>' | source ID: 3 | target ID: 3
'eng_Latn' | source ID: 4 | target ID: None
'hin_Deva' | source ID: 8 | target ID: None
'mar_Deva' | source ID: 18 | target ID: None
'sat_Olck' | source ID: 29925 | target ID: None


In [48]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Use one real example from the prepared dataset.
src_col, tgt_col = language_columns["santali"]
row = all_data["santali"]["train"].iloc[0]

source_text = str(row[src_col])
target_text = str(row[tgt_col])

# Apply the same language-conditioning convention used earlier.
processed_source = processor.preprocess_batch(
    [source_text],
    src_lang="eng_Latn",
    tgt_lang="sat_Olck",
)[0]

source_encoding = tok(
    processed_source,
    return_tensors="pt",
    add_special_tokens=True,
)

target_encoding = tok(
    text_target=target_text,
    return_tensors="pt",
    add_special_tokens=True,
)

print("=== Model embedding sizes ===")
print("Encoder embedding rows:", model.get_input_embeddings().num_embeddings)
print("Decoder embedding rows:", model.get_decoder().embed_tokens.num_embeddings)
print("Output projection rows:", model.get_output_embeddings().out_features)

print("\n=== Conditioned source ===")
print(processed_source[:300])
print("Source IDs:", source_encoding["input_ids"][0, :12].tolist())
print("Source length:", source_encoding["input_ids"].shape[1])

print("\n=== Target labels ===")
print("Target text:", target_text[:200])
print("Target IDs:", target_encoding["input_ids"][0, :12].tolist())
print("Target length:", target_encoding["input_ids"].shape[1])

print("\n=== ID range checks ===")
print(
    "All source IDs valid:",
    source_encoding["input_ids"].min().item() >= 0
    and source_encoding["input_ids"].max().item()
    < model.get_input_embeddings().num_embeddings
)
print(
    "All target IDs valid:",
    target_encoding["input_ids"].min().item() >= 0
    and target_encoding["input_ids"].max().item()
    < model.get_output_embeddings().out_features
)

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

=== Model embedding sizes ===
Encoder embedding rows: 122706
Decoder embedding rows: 122672
Output projection rows: 122672

=== Conditioned source ===
eng_Latn sat_Olck Instacart 's Shopper app looks like a green Instacart bag on a white background .
Source IDs: [4, 29925, 57861, 18782, 38095, 40, 1888, 59633, 30179, 42833, 91069, 51927]
Source length: 24

=== Target labels ===
Target text: ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾
Target IDs: [95037, 72860, 90936, 95244, 78321, 97232, 102588, 103847, 78143, 98715, 48377, 103575]
Target length: 28

=== ID range checks ===
All source IDs valid: True
All target IDs valid: True


In [49]:
import inspect
from transformers import AutoConfig, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("=== Forward method signature ===")
print(inspect.signature(model.forward))

print("\n=== Generation configuration ===")
print(model.generation_config)

print("\n=== Forward method source ===")
try:
    print(inspect.getsource(type(model).forward))
except (OSError, TypeError) as exc:
    print("Could not retrieve source:", repr(exc))

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

=== Forward method signature ===
(input_ids: Optional[torch.LongTensor] = None, attention_mask: Optional[torch.Tensor] = None, decoder_input_ids: Optional[torch.LongTensor] = None, decoder_attention_mask: Optional[torch.LongTensor] = None, head_mask: Optional[torch.Tensor] = None, decoder_head_mask: Optional[torch.Tensor] = None, cross_attn_head_mask: Optional[torch.Tensor] = None, encoder_outputs: Optional[Tuple[Tuple[torch.FloatTensor]]] = None, past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None, inputs_embeds: Optional[torch.FloatTensor] = None, decoder_inputs_embeds: Optional[torch.FloatTensor] = None, labels: Optional[torch.LongTensor] = None, use_cache: Optional[bool] = None, output_attentions: Optional[bool] = None, output_hidden_states: Optional[bool] = None, return_dict: Optional[bool] = None) -> Union[Tuple[torch.Tensor], transformers.modeling_outputs.Seq2SeqLMOutput]

=== Generation configuration ===
GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_tok


# 6. Training Pipeline Validation

## 6.1 Forward Pass and Gradient Verification

### Objective

Before implementing LoRA fine-tuning, we need to verify that the pretrained IndicTrans2 model can process our prepared parallel data, calculate a valid translation loss, and propagate gradients successfully.

### Procedure

Using one real Santali training example, we will:

1. Prepare the English source sentence using the established `eng_Latn` → `sat_Olck` conditioning convention.
2. Tokenize the source and target sentences.
3. Verify that target token IDs are within the decoder vocabulary range.
4. Run a forward pass and check that the loss is finite.
5. Run a backward pass and confirm that gradients are computed.

### Expected Outcome

The model should produce a finite loss, correctly shaped logits, and a successful backward pass.

This is a diagnostic test only. We will not perform an optimizer step or update the pretrained weights.

Once these checks pass, we can inspect the model's actual attention-module names and determine suitable LoRA target modules before beginning baseline fine-tuning.


In [50]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

model.train()

src_col, tgt_col = language_columns["santali"]
row = all_data["santali"]["train"].iloc[0]

source_text = str(row[src_col])
target_text = str(row[tgt_col])

processed_source = processor.preprocess_batch(
    [source_text],
    src_lang="eng_Latn",
    tgt_lang="sat_Olck",
)[0]

source_batch = tok(
    processed_source,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

target_batch = tok(
    text_target=target_text,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

input_ids = source_batch["input_ids"].to(DEVICE)
attention_mask = source_batch["attention_mask"].to(DEVICE)
labels = target_batch["input_ids"].to(DEVICE)

print("Input shape:", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))
print("Labels min/max:", labels.min().item(), labels.max().item())
print("Decoder vocabulary size:", model.config.decoder_vocab_size)

assert labels.min().item() >= 0
assert labels.max().item() < model.config.decoder_vocab_size

model.zero_grad(set_to_none=True)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

print("\nForward-pass loss:", outputs.loss.item())
print("Loss finite:", torch.isfinite(outputs.loss).item())
print("Logits shape:", tuple(outputs.logits.shape))

assert torch.isfinite(outputs.loss).item(), "Loss is not finite!"

outputs.loss.backward()

gradient_count = sum(
    parameter.grad is not None
    for parameter in model.parameters()
)

print("Parameters receiving gradients:", gradient_count)
print("Backward pass completed successfully.")

del outputs, model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Input shape: (1, 24)
Labels shape: (1, 28)
Labels min/max: 2 111525
Decoder vocabulary size: 122672

Forward-pass loss: 1.7412109375
Loss finite: True
Logits shape: (1, 28, 122672)
Parameters receiving gradients: 763
Backward pass completed successfully.


In [52]:
import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

attention_modules = []

for name, module in model.named_modules():
    class_name = module.__class__.__name__.lower()

    if "attention" in class_name or "attn" in class_name:
        attention_modules.append((name, module.__class__.__name__))

print("Attention-related modules:")
for name, class_name in attention_modules:
    print(f"{name}: {class_name}")

print("\nTotal attention-related modules:", len(attention_modules))

Attention-related modules:
model.encoder.layers.0.self_attn: IndicTransAttention
model.encoder.layers.1.self_attn: IndicTransAttention
model.encoder.layers.2.self_attn: IndicTransAttention
model.encoder.layers.3.self_attn: IndicTransAttention
model.encoder.layers.4.self_attn: IndicTransAttention
model.encoder.layers.5.self_attn: IndicTransAttention
model.encoder.layers.6.self_attn: IndicTransAttention
model.encoder.layers.7.self_attn: IndicTransAttention
model.encoder.layers.8.self_attn: IndicTransAttention
model.encoder.layers.9.self_attn: IndicTransAttention
model.encoder.layers.10.self_attn: IndicTransAttention
model.encoder.layers.11.self_attn: IndicTransAttention
model.encoder.layers.12.self_attn: IndicTransAttention
model.encoder.layers.13.self_attn: IndicTransAttention
model.encoder.layers.14.self_attn: IndicTransAttention
model.encoder.layers.15.self_attn: IndicTransAttention
model.encoder.layers.16.self_attn: IndicTransAttention
model.encoder.layers.17.self_attn: IndicTransAtt

In [53]:
# Inspect the actual linear layers in one encoder and one decoder attention module.

for module_name in [
    "model.encoder.layers.0.self_attn",
    "model.decoder.layers.0.self_attn",
    "model.decoder.layers.0.encoder_attn",
]:
    module = model.get_submodule(module_name)

    print(f"\n{module_name}")
    print("-" * len(module_name))

    for name, submodule in module.named_modules():
        if isinstance(submodule, torch.nn.Linear):
            print(
                f"{name or '<self>'}: "
                f"{submodule.in_features} -> {submodule.out_features}"
            )


model.encoder.layers.0.self_attn
--------------------------------
k_proj: 1024 -> 1024
v_proj: 1024 -> 1024
q_proj: 1024 -> 1024
out_proj: 1024 -> 1024

model.decoder.layers.0.self_attn
--------------------------------
k_proj: 1024 -> 1024
v_proj: 1024 -> 1024
q_proj: 1024 -> 1024
out_proj: 1024 -> 1024

model.decoder.layers.0.encoder_attn
-----------------------------------
k_proj: 1024 -> 1024
v_proj: 1024 -> 1024
q_proj: 1024 -> 1024
out_proj: 1024 -> 1024


In [55]:
import torch
import transformers
import peft
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

for package in ["torchao", "accelerate", "tokenizers"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

PyTorch: 2.11.0+cu128
Transformers: 4.46.3
PEFT: 0.20.0
torchao: 0.10.0
accelerate: 1.14.0
tokenizers: 0.20.3


In [56]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [57]:
from peft import LoraConfig, get_peft_model

# Apply LoRA to query and value projections across encoder and decoder attention.
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

lora_model = get_peft_model(model, LORA_CONFIG)

lora_model.print_trainable_parameters()

trainable_params = sum(
    p.numel() for p in lora_model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in lora_model.parameters())

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

# Verify exactly which modules received LoRA adapters.
print("\nLoRA-wrapped modules:")
for name, module in lora_model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        print(name)

trainable params: 3,538,944 || all params: 1,211,635,712 || trainable%: 0.2921

Trainable parameters: 3,538,944
Total parameters: 1,211,635,712
Trainable percentage: 0.2921%

LoRA-wrapped modules:
base_model.model.model.encoder.layers.0.self_attn.v_proj
base_model.model.model.encoder.layers.0.self_attn.q_proj
base_model.model.model.encoder.layers.1.self_attn.v_proj
base_model.model.model.encoder.layers.1.self_attn.q_proj
base_model.model.model.encoder.layers.2.self_attn.v_proj
base_model.model.model.encoder.layers.2.self_attn.q_proj
base_model.model.model.encoder.layers.3.self_attn.v_proj
base_model.model.model.encoder.layers.3.self_attn.q_proj
base_model.model.model.encoder.layers.4.self_attn.v_proj
base_model.model.model.encoder.layers.4.self_attn.q_proj
base_model.model.model.encoder.layers.5.self_attn.v_proj
base_model.model.model.encoder.layers.5.self_attn.q_proj
base_model.model.model.encoder.layers.6.self_attn.v_proj
base_model.model.model.encoder.layers.6.self_attn.q_proj
base_

In [58]:
# Verify trainable parameter names and confirm the base model is frozen.

trainable_names = [
    name
    for name, param in lora_model.named_parameters()
    if param.requires_grad
]

print("Number of trainable parameter tensors:", len(trainable_names))
print("\nFirst 20 trainable parameter names:")

for name in trainable_names[:20]:
    print(name)

unexpected_trainable = [
    name for name in trainable_names
    if "lora_A" not in name and "lora_B" not in name
]

print("\nUnexpected trainable parameters:", len(unexpected_trainable))

assert len(trainable_names) > 0
assert len(unexpected_trainable) == 0, (
    "Some trainable parameters are not LoRA A/B adapters."
)

print("\nPASS: Only LoRA adapter parameters are trainable.")

Number of trainable parameter tensors: 216

First 20 trainable parameter names:
base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.encoder.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.encoder.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.model.encoder.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.encoder.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.encoder.layers.2.self_attn.v_proj.lora_A.default.weight
base_model.model.model.encoder.layers.2.self_attn.v_proj.lora_B.default.weight
base_model.model.model.encoder.layers.2.self_attn.q_proj.lora_A.default.weight
base_model.model.model.encoder.layers.2.self_attn.q

In [59]:
# Prepare Santali training and validation examples for LoRA fine-tuning.

from torch.utils.data import Dataset, DataLoader
import pandas as pd

LANGUAGE = "santali"
SRC_LANG = "eng_Latn"
TGT_LANG = "sat_Olck"
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

src_col, tgt_col = language_columns[LANGUAGE]

train_df = all_data[LANGUAGE]["train"].copy()
val_df = all_data[LANGUAGE]["validation"].copy()

def prepare_examples(df):
    examples = []

    for _, row in df.iterrows():
        source_text = str(row[src_col]).strip()
        target_text = str(row[tgt_col]).strip()

        if not source_text or not target_text:
            continue

        processed_source = processor.preprocess_batch(
            [source_text],
            src_lang=SRC_LANG,
            tgt_lang=TGT_LANG,
        )[0]

        examples.append({
            "source": processed_source,
            "target": target_text,
        })

    return examples

train_examples = prepare_examples(train_df)
val_examples = prepare_examples(val_df)

print("Language:", LANGUAGE)
print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("\nPrepared example:")
print("Source:", train_examples[0]["source"])
print("Target:", train_examples[0]["target"])

assert len(train_examples) > 0
assert len(val_examples) > 0

Language: santali
Training examples: 17883
Validation examples: 993

Prepared example:
Source: eng_Latn sat_Olck Instacart 's Shopper app looks like a green Instacart bag on a white background .
Target: ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾


In [60]:
import torch
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 4

class TranslationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_translation_batch(examples):
    sources = [example["source"] for example in examples]
    targets = [example["target"] for example in examples]

    source_batch = tok(
        sources,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    )

    target_batch = tok(
        text_target=targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt",
    )

    labels = target_batch["input_ids"].clone()
    labels[target_batch["attention_mask"] == 0] = -100

    return {
        "input_ids": source_batch["input_ids"],
        "attention_mask": source_batch["attention_mask"],
        "labels": labels,
    }


train_dataset = TranslationDataset(train_examples)
val_dataset = TranslationDataset(val_examples)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")

print("\nLabels contain -100:", (batch["labels"] == -100).any().item())
print("Input IDs within source vocabulary:",
      batch["input_ids"].max().item() < model.config.encoder_vocab_size)

valid_labels = batch["labels"][batch["labels"] != -100]
print("Target IDs within decoder vocabulary:",
      valid_labels.max().item() < model.config.decoder_vocab_size)

assert batch["input_ids"].shape[0] == BATCH_SIZE
assert (batch["labels"] == -100).any().item()
assert batch["input_ids"].max().item() < model.config.encoder_vocab_size
assert valid_labels.max().item() < model.config.decoder_vocab_size

print("\nPASS: Batch construction and vocabulary checks succeeded.")

input_ids: shape=(4, 52), dtype=torch.int64
attention_mask: shape=(4, 52), dtype=torch.int64
labels: shape=(4, 44), dtype=torch.int64

Labels contain -100: True
Input IDs within source vocabulary: True
Target IDs within decoder vocabulary: True

PASS: Batch construction and vocabulary checks succeeded.


In [61]:
from torch.optim import AdamW

# Keep the first test small; this is not the full training loop.
LEARNING_RATE = 1e-4

optimizer = AdamW(
    (p for p in lora_model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
)

lora_model.train()
lora_model.zero_grad(set_to_none=True)

batch = {
    key: value.to(DEVICE)
    for key, value in batch.items()
}

# Snapshot one trainable parameter before the update.
tracked_name, tracked_param = next(
    (name, param)
    for name, param in lora_model.named_parameters()
    if param.requires_grad and "lora_B" in name
)
parameter_before = tracked_param.detach().clone()

outputs = lora_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
    use_cache=False,
)

loss = outputs.loss
print("Loss before update:", loss.item())
assert torch.isfinite(loss).item(), "Loss is not finite."

loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in lora_model.parameters() if p.requires_grad],
    max_norm=1.0,
)
print("Gradient norm:", grad_norm.item())
assert torch.isfinite(grad_norm).item(), "Gradient norm is not finite."

optimizer.step()

parameter_after = dict(lora_model.named_parameters())[tracked_name].detach()
changed = not torch.equal(parameter_before, parameter_after)

print("Tracked parameter:", tracked_name)
print("Parameter changed after optimizer step:", changed)

assert changed, "The tracked LoRA parameter did not change."
print("\nPASS: One optimizer step completed successfully.")

Loss before update: 2.49609375
Gradient norm: 0.4363577365875244
Tracked parameter: base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight
Parameter changed after optimizer step: True

PASS: One optimizer step completed successfully.



# 7. Baseline LoRA Fine-Tuning

## 7.1 Experimental Setup

We will establish the baseline by fine-tuning the pretrained IndicTrans2 model using LoRA while keeping its original vocabulary unchanged.

The first experiment will use English-to-Santali translation, with the established language-tag convention and the prepared training and validation splits.

### Training Configuration

- **Maximum epochs:** 10
- **Early stopping:** Patience of 2 epochs
- **Early-stopping minimum improvement:** 0.001 in validation loss
- **Micro-batch size:** 4
- **Gradient accumulation steps:** 4
- **Effective batch size:** 16
- **Learning rate:** 0.0001
- **Optimizer:** AdamW
- **LoRA rank:** 16
- **LoRA alpha:** 32
- **LoRA dropout:** 0.05
- **Vocabulary:** Original, unchanged

### Experimental Procedure

1. Reload the original pretrained checkpoint and initialize fresh LoRA adapters.
2. Train on the Santali training split.
3. Evaluate validation loss after each epoch.
4. Save the best checkpoint and stop early if validation loss stops improving.
5. Record training loss, validation loss, runtime, and the best epoch.
6. Evaluate the selected checkpoint on the held-out test split only after training and model selection are complete.

This trained baseline will serve as the reference for the subsequent vocabulary-extension experiment. The same data splits, language tags, LoRA configuration, and evaluation procedure must be used in both experiments.


In [63]:
# Santali baseline LoRA training configuration

import random
import numpy as np
import torch

SEED = 42
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 10                 # Maximum epochs
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Experiment: Santali baseline — original vocabulary + LoRA")
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Minimum validation-loss improvement:", EARLY_STOPPING_MIN_DELTA)
print("Micro-batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Warmup ratio:", WARMUP_RATIO)
print("Device:", DEVICE)

Experiment: Santali baseline — original vocabulary + LoRA
Maximum epochs: 10
Early stopping patience: 2
Minimum validation-loss improvement: 0.001
Micro-batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 0.0001
Warmup ratio: 0.03
Device: cuda


In [64]:
import gc
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

# Release previous model instances and GPU memory.
if "lora_model" in globals():
    del lora_model
if "model" in globals():
    del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load a fresh copy of the original pretrained checkpoint.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

# Create new LoRA adapters with the established configuration.
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

lora_model = get_peft_model(model, LORA_CONFIG)

# Verify the fresh training setup.
lora_model.print_trainable_parameters()

print("\nModel:", MODEL_ID)
print("Device:", DEVICE)
print("Maximum epochs:", NUM_EPOCHS)
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Original vocabulary retained:", True)
print("Fresh LoRA adapters initialized:", True)

trainable params: 3,538,944 || all params: 1,211,635,712 || trainable%: 0.2921

Model: ai4bharat/indictrans2-indic-indic-1B
Device: cuda
Maximum epochs: 10
Early-stopping patience: 2
Original vocabulary retained: True
Fresh LoRA adapters initialized: True


In [65]:
import math
from pathlib import Path
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Reproducible baseline output directory
BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

# Number of optimizer updates in one epoch
updates_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)
total_training_steps = updates_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

optimizer = AdamW(
    (p for p in lora_model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Training batches per epoch:", len(train_loader))
print("Optimizer updates per epoch:", updates_per_epoch)
print("Maximum optimizer updates:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Checkpoint directory:", BASELINE_DIR)

assert len(train_loader) > 0
assert len(val_loader) > 0
assert total_training_steps > 0

print("\nPASS: Optimizer and scheduler are ready.")

Training batches per epoch: 4471
Optimizer updates per epoch: 1118
Maximum optimizer updates: 11180
Warmup steps: 335
Initial learning rate: 0.0
Checkpoint directory: /kaggle/working/santali_baseline_lora

PASS: Optimizer and scheduler are ready.



## 7.3 Baseline Training Loop

Train the fresh LoRA adapters on the Santali training split and evaluate validation loss after every epoch.

The loop will:
- Accumulate gradients over four micro-batches.
- Clip gradients and update the optimizer and learning-rate scheduler.
- Save the adapter checkpoint whenever validation loss improves by at least the configured minimum delta.
- Stop after two consecutive epochs without sufficient improvement, or after ten epochs.
- Restore the best checkpoint at the end.

The test split will remain untouched until final evaluation.


In [ ]:
import copy
import json
import time
import numpy as np
import torch
from pathlib import Path

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []
global_step = 0

best_checkpoint_dir = BASELINE_DIR / "best_adapter"
best_checkpoint_dir.mkdir(parents=True, exist_ok=True)

optimizer.zero_grad(set_to_none=True)
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    lora_model.train()
    train_losses = []
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = lora_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            use_cache=False,
        )

        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, batch {batch_idx}"
            )

        train_losses.append(loss.item())
        (loss / GRADIENT_ACCUMULATION_STEPS).backward()

        should_update = (
            batch_idx % GRADIENT_ACCUMULATION_STEPS == 0
            or batch_idx == len(train_loader)
        )

        if should_update:
            torch.nn.utils.clip_grad_norm_(
                [p for p in lora_model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if batch_idx % 500 == 0:
            print(
                f"Epoch {epoch}/{NUM_EPOCHS} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    # Validation: no gradients or optimizer updates.
    lora_model.eval()
    val_losses = []

    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = {k: v.to(DEVICE) for k, v in val_batch.items()}

            val_outputs = lora_model(
                input_ids=val_batch["input_ids"],
                attention_mask=val_batch["attention_mask"],
                labels=val_batch["labels"],
                use_cache=False,
            )

            val_loss = val_outputs.loss
            if not torch.isfinite(val_loss):
                raise RuntimeError(
                    f"Non-finite validation loss at epoch {epoch}"
                )

            val_losses.append(val_loss.item())

    mean_train_loss = float(np.mean(train_losses))
    mean_val_loss = float(np.mean(val_losses))
    elapsed = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "validation_loss": mean_val_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "optimizer_steps": global_step,
        "epoch_seconds": elapsed,
    }
    history.append(record)

    print(
        f"\nEpoch {epoch}: "
        f"train_loss={mean_train_loss:.4f}, "
        f"validation_loss={mean_val_loss:.4f}, "
        f"time={elapsed / 60:.1f} min"
    )

    # A new best requires the configured minimum improvement.
    if mean_val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = mean_val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        lora_model.save_pretrained(best_checkpoint_dir)

        with open(BASELINE_DIR / "training_history.json", "w") as f:
            json.dump(history, f, indent=2)

        print(f"Saved new best adapter at epoch {epoch}.")
    else:
        epochs_without_improvement += 1
        print(
            "No sufficient validation improvement: "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

# Restore the best adapter weights before evaluation.
if best_epoch == 0:
    raise RuntimeError("No best checkpoint was saved.")

from peft import PeftModel

lora_model.load_adapter(str(best_checkpoint_dir), adapter_name="best")
lora_model.set_adapter("best")

with open(BASELINE_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Optimizer steps completed:", global_step)
print("Total training time (minutes):", round((time.time() - training_start) / 60, 2))
print("Best adapter directory:", best_checkpoint_dir)

In [ ]:
from pathlib import Path
import json

BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BEST_DIR = BASELINE_DIR / "best_adapter"

print("Baseline directory exists:", BASELINE_DIR.exists())
print("Best adapter directory exists:", BEST_DIR.exists())

if BASELINE_DIR.exists():
    print("\nFiles in baseline directory:")
    for path in sorted(BASELINE_DIR.rglob("*")):
        if path.is_file():
            print(f"  {path.relative_to(BASELINE_DIR)} ({path.stat().st_size / 1024:.1f} KB)")

history_path = BASELINE_DIR / "training_history.json"

if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)

    print("\nRecorded training history:")
    for record in history:
        print(record)
else:
    print("\nNo training_history.json found.")